# Reliable evaluation of crime forecasting: benchmark audit, reproduction gap and calibrated intervals
### Experiments for the journal paper (Option A)

This notebook runs every experiment of the paper in one go and saves all tables, figures and logs.

| Part | Question | Data |
|---|---|---|
| **I. Benchmark audit** | Do published deep crime models reproduce? Does the benchmark's official metric measure forecasting skill? How do rankings change under a full protocol? | Public NYC-/Chicago-Crimes benchmark (DeepCrime, ST-SHN, ST-HSL) |
| **II. Weekly forecasting, three cities** | Does a global gradient-boosted Poisson model beat naive, statistical and deep forecasters, over several seeds and horizons 1–4 weeks? | Chicago, New York, San Francisco open data |
| **III. Calibrated intervals** | Which interval method keeps 90 % coverage for quiet *and* busy series: parametric, split conformal, **Mondrian**, CQR, locally weighted, adaptive (ACI)? | same |
| **IV. Rolling origins and shift** | Do the conclusions hold in every year, including the 2020 shift? | test years 2020–2024 + main split |
| **V. Fairness** | Are coverage and errors equal across poorer / richer and minority / white-majority areas? | ACS 5-year census tracts mapped to areas |

**Pre-registered hypotheses** (judged automatically in the last section; thresholds fixed before running)
- **A1 Reproduction gap.** Re-running the official ST-HSL and ST-SHN code (validation-selected, mean over seeds) gives a higher official-protocol MAE than the published number in at least 6 of the 8 city × category cells, for each model.
- **A2 Metric exploit.** Under the official protocol (crime days only) a constant forecast of 1 beats the published ST-HSL MAPE in at least one category in each city, and the protocol-aware version of our forecaster (zero-truncated Poisson median) beats the published ST-HSL MAE in at least one category in each city.
- **A3 Ranking instability.** Model rankings under the official and the full protocol (all cell-days) disagree: Kendall τ < 0.8 in at least one city, or a different best model.
- **A4 Simple beats deep.** Our gradient-boosted Poisson forecaster has a lower full-protocol Poisson deviance than every re-run deep model (ST-HSL, ST-SHN, Graph WaveNet, MTGNN) in both benchmark cities, in every seed.
- **A5 Forecast skill.** On the weekly panels of all three cities the validation-selected forecaster beats every naive and statistical baseline at horizons 1–4 (week-block bootstrap CI of the deviance gain above 0, in every seed).
- **A6 Mondrian calibration.** Mondrian conformal intervals keep every series-volume quintile within 3 points of 90 % in every city and horizon, while standard split conformal, Poisson and negative-binomial intervals do not.
- **A7 Over time.** A6 holds in every rolling-origin test year, including 2020; in 2020 the adaptive (ACI) variant brings overall coverage within 2 points of 90 %.
- **A8 Equity.** Mondrian intervals have a smaller coverage gap between the poorest and the richest third of areas than standard split conformal, in every city.

**How to run on Kaggle.** *Settings → Accelerator → GPU T4 x2*, *Internet → On*, then **Save Version → Save & Run All (Commit)** (a commit keeps running when the browser is closed). Expected run time 6–9 hours (Kaggle's limit is 12 h). Every part saves its own tables as soon as it finishes, and the zip file `/kaggle/working/optionA_outputs.zip` is refreshed after every part, so a late failure never loses earlier results. If one part fails the notebook logs the error and carries on with the next part. Order of execution: benchmark code audit → city data → forecasters → intervals → rolling origins → equity check → **benchmark re-runs of the deep models last** (their run time is the least predictable; extra seeds are skipped automatically when the time budget runs low) → verdicts.

**What to send back:** `optionA_outputs.zip` from the *Output* tab (it contains `tables/`, `figures/`, `logs/` and `results_summary.md`).

**Responsible use.** All models forecast counts for places and weeks, never for people. Census variables are used only as area-level aggregates to check whether forecast quality is equal across communities.

In [ ]:
import os, sys, time, json, math, gc, re, warnings, zipfile, shutil, subprocess, random, traceback, hashlib, io, contextlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
T0 = time.time()

# ------------------------------------------------------------------ configuration
FAST        = os.environ.get("OA_FAST", "0") == "1"          # quick smoke test only
TIME_LIMIT_H = 11.4                                           # stop optional work before Kaggle's 12-hour limit
START_YEAR  = int(os.environ.get("OA_START_YEAR", "2016"))
TRAIN_END   = pd.Timestamp("2023-12-31")                      # same split as the capstone
VAL_END     = pd.Timestamp("2024-06-30")                      # calibration / validation block; test = after this date
CITIES      = ["CHI", "NYC", "SF"]
CITY_NAME   = {"CHI": "Chicago", "NYC": "New York", "SF": "San Francisco"}
DROP_LAST_WEEKS = {"CHI": 1, "NYC": 4, "SF": 2}               # late-reported records make the newest weeks incomplete
HORIZONS    = [1, 2, 3, 4]
NN_HORIZONS = [1, 4]                                          # neural forecasters (slower) are run for these horizons
SEEDS_TREE  = [0] if FAST else [0, 1, 2]
SEEDS_NN    = [0] if FAST else [0, 1, 2]
SEEDS_BENCH = [0] if FAST else [0, 1, 2]                      # ST-HSL, Graph WaveNet, MTGNN, our benchmark forecaster
SEEDS_STSHN = [0] if FAST else [0, 1]                         # ST-SHN (TensorFlow, batch size 1, slower)
ROLL_YEARS  = [2021, 2022] if FAST else [2020, 2021, 2022, 2023, 2024]
ALPHA       = 0.10                                            # 90 % intervals
SEED        = 42

KAGGLE = Path("/kaggle/working").exists()
WORK   = Path("/kaggle/working") if KAGGLE else Path(os.environ.get("OA_WORK", Path.cwd()))
OUT    = WORK / "optionA_outputs"; FIG = OUT / "figures"; TAB = OUT / "tables"; LOGS = OUT / "logs"; PRED = OUT / "predictions"
DATA   = WORK / "raw_data"; REPOS = WORK / "repos"
for p in (OUT, FIG, TAB, LOGS, PRED, DATA, REPOS): p.mkdir(parents=True, exist_ok=True)

def hours_used(): return (time.time() - T0) / 3600
def time_left(): return TIME_LIMIT_H - hours_used()
def log(msg):
    line = f"[{hours_used():5.2f} h] {msg}"; print(line, flush=True)
    with open(LOGS / "progress.log", "a") as f: f.write(line + "\n")
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG / f"{name}.png", dpi=140, bbox_inches="tight"); plt.show(); plt.close("all")
def save_table(df, name, index=False):
    df.to_csv(TAB / f"{name}.csv", index=index); return df
def refresh_zip():
    """Zip everything in the output folder (called after every part)."""
    dest = WORK / "optionA_outputs.zip"; tmp = WORK / "optionA_outputs.zip.part"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in OUT.rglob("*"):
            if p.is_file(): z.write(p, "optionA_outputs/" + str(p.relative_to(OUT)))
    tmp.replace(dest); log(f"zip refreshed: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")

STATUS = {}
@contextlib.contextmanager
def part(name):
    """Run one part; on an error write the traceback to logs/errors.log and continue with the next part."""
    t = time.time(); log(f"===== {name} started"); STATUS[name] = "running"
    try:
        yield; STATUS[name] = f"ok ({(time.time() - t) / 60:.0f} min)"
    except Exception:
        tb = traceback.format_exc(); STATUS[name] = "FAILED – see logs/errors.log"
        print(tb); open(LOGS / "errors.log", "a").write(f"===== {name}\n{tb}\n")
    finally:
        log(f"===== {name}: {STATUS[name]}"); json.dump(STATUS, open(LOGS / "status.json", "w"), indent=1)
        try: refresh_zip()
        except Exception as e: print("zip failed:", e)

def pip_install(*pkgs):
    for p in pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], capture_output=True, timeout=900)
try: import lightgbm as lgb
except ImportError: pip_install("lightgbm"); import lightgbm as lgb
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy.stats import poisson as _pois, nbinom as _nb, kendalltau, spearmanr

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()

def pick_device():
    if torch.cuda.is_available():
        try: torch.ones(8, device="cuda").sum().item(); return torch.device("cuda")
        except Exception as e: print("CUDA present but unusable with this PyTorch build:", str(e)[:120])
    return torch.device("cpu")
DEVICE = pick_device()
def xgb_device():
    try:
        b = xgb.XGBRegressor(n_estimators=2, device="cuda", tree_method="hist"); b.fit(np.random.rand(64, 3), np.random.rand(64)); return "cuda"
    except Exception: return "cpu"
XGB_DEVICE = xgb_device() if DEVICE.type == "cuda" else "cpu"
VERSIONS = dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__, torch=torch.__version__, xgboost=xgb.__version__,
                lightgbm=lgb.__version__, device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none", fast=FAST)
json.dump(VERSIONS, open(LOGS / "environment.json", "w"), indent=1)
log(f"environment: {VERSIONS}")
if DEVICE.type == "cpu" and not FAST: print("WARNING: no usable GPU. Switch the accelerator to 'GPU T4 x2', or expect a much longer run.")

# ------------------------------------------------------------------ shared metric helpers
def pdev(y, mu):
    mu = np.maximum(np.asarray(mu, float), 1e-6); y = np.asarray(y, float)
    return 2 * np.mean(np.where(y > 0, y * np.log(np.maximum(y, 1e-12) / mu), 0) - (y - mu))
def unit_dev(y, mu):
    mu = np.maximum(np.asarray(mu, float), 1e-6); y = np.asarray(y, float)
    return 2 * (np.where(y > 0, y * np.log(np.maximum(y, 1e-12) / mu), 0) - (y - mu))
def block_boot(diff, blocks, B=1000, seed=SEED):
    """Mean of `diff` and a 95 % CI from a bootstrap that resamples whole blocks (weeks or days)."""
    wk = pd.factorize(np.asarray(blocks))[0]; s = np.bincount(wk, diff); n = np.bincount(wk); r = np.random.default_rng(seed)
    idx = r.integers(0, len(s), (B, len(s))); est = s[idx].sum(1) / n[idx].sum(1)
    return float(np.mean(diff)), float(np.percentile(est, 2.5)), float(np.percentile(est, 97.5))

# ------------------------------------------------------------------ shared forecasters (gradient-boosted Poisson models)
def fit_lgb_pois(Xtr, ytr, Xes, yes, seed=0, n_est=None):
    m = lgb.LGBMRegressor(objective="poisson", n_estimators=n_est or (150 if FAST else 3000), learning_rate=0.05, num_leaves=127, min_child_samples=20,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m
def fit_xgb_pois(Xtr, ytr, Xes, yes, seed=0):
    m = xgb.XGBRegressor(objective="count:poisson", n_estimators=150 if FAST else 2000, learning_rate=0.05, max_depth=7, subsample=0.8,
                         colsample_bytree=0.8, min_child_weight=5, tree_method="hist", device=XGB_DEVICE, enable_categorical=True,
                         max_cat_to_onehot=1, random_state=seed, early_stopping_rounds=50, eval_metric="poisson-nloglik")
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], verbose=False); return m
def fit_lgb_quantile(Xtr, ytr, Xes, yes, q, seed=0):
    m = lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=150 if FAST else 2000, learning_rate=0.05, num_leaves=63, min_child_samples=50,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m
def fit_lgb_l2(Xtr, ytr, Xes, yes, seed=0):
    m = lgb.LGBMRegressor(objective="l2", n_estimators=150 if FAST else 2000, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=seed, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m

# ------------------------------------------------------------------ shared interval helpers (all calibrated on a held-out block only)
def conformal_q(scores, alpha=ALPHA):
    scores = np.asarray(scores, float); n = len(scores)
    if n == 0: return np.inf
    lev = np.ceil((n + 1) * (1 - alpha)) / n
    return np.inf if lev > 1 else float(np.quantile(scores, lev, method="higher"))
def norm_score(y, mu): return np.abs(y - mu) / np.sqrt(mu + 0.5)
def int_from_q(mu, q, scale=None):
    s = np.sqrt(mu + 0.5) if scale is None else scale
    lo = np.maximum(np.ceil(mu - q * s), 0); hi = np.floor(mu + q * s)
    bad = hi < lo; r = np.round(mu); return np.where(bad, r, lo), np.where(bad, r, hi)
def mondrian_edges(mu_cal, k=5): return np.unique(np.quantile(mu_cal, np.linspace(0, 1, k + 1)[1:-1]))
def nb_size(mu, y):                                            # negative-binomial size by moments: Var = mu + mu^2 / r
    return float(min(max(np.sum(mu ** 2) / max(np.sum((y - mu) ** 2 - mu), 1e-6), 1e-3), 1e4))
def parametric_interval(mu, kind, r=None):
    mu = np.maximum(mu, 1e-6)
    if kind == "Poisson": return _pois.ppf(ALPHA / 2, mu), _pois.ppf(1 - ALPHA / 2, mu)
    p = r / (r + mu); return _nb.ppf(ALPHA / 2, r, p), _nb.ppf(1 - ALPHA / 2, r, p)
def interval_score(y, lo, hi): return (hi - lo) + (2 / ALPHA) * np.maximum(lo - y, 0) + (2 / ALPHA) * np.maximum(y - hi, 0)
def coverage_report(y, lo, hi, vq=None, area=None):
    """Overall coverage, width, interval score, coverage per volume quintile and per area."""
    cov = (y >= lo) & (y <= hi); out = dict(coverage=cov.mean(), width=(hi - lo).mean(), interval_score=interval_score(y, lo, hi).mean())
    if vq is not None:
        for q in range(1, 6): out[f"cov_Q{q}"] = cov[vq == q].mean() if (vq == q).any() else np.nan
        out["max_abs_dev_from_90"] = np.nanmax([abs(out[f"cov_Q{q}"] - (1 - ALPHA)) for q in range(1, 6)])
        out["gap_Q5_minus_Q1"] = out["cov_Q5"] - out["cov_Q1"]
    if area is not None:
        ac = pd.Series(cov).groupby(np.asarray(area)).mean()
        out["areas_within_5pts"] = float((ac.sub(1 - ALPHA).abs() <= 0.05).mean()); out["worst_area_dev"] = float(ac.sub(1 - ALPHA).abs().max())
        out["sd_area_coverage"] = float(ac.std())
    return out

## Part I. Audit of the public NYC-/Chicago-Crimes benchmark
Most deep crime-prediction papers (DeepCrime, ST-SHN, ST-HSL and later work) report results on one public benchmark: daily counts of four crime categories on a 3 km grid (New York 2014–2015, 16 × 16 cells: burglary, larceny, robbery, assault; Chicago 2016–2017, 14 × 12 cells: theft, battery, assault, damage), with 608/609 training, 30 validation and 92 test days.

This part:
1. downloads the benchmark and the **official code** of ST-HSL (PyTorch) and ST-SHN (TensorFlow 1), and the official model code of Graph WaveNet and MTGNN, and records the exact commit of each;
2. **audits the evaluation code** automatically (which cell-days are scored, how the reported epoch is chosen);
3. trains our gradient-boosted Poisson forecaster and simple baselines, **re-runs ST-HSL and ST-SHN with their own code** and trains Graph WaveNet and MTGNN with their official model code under the ST-HSL training rule, each with several seeds;
4. scores every model under three protocols: **P1 official ST-HSL** (crime days only), **P2 official ST-SHN** (crime days in cells active on 20–80 % of training days) and **P3 full** (all cell-days: MAE, RMSE, Poisson deviance);
5. compares re-runs with the **published** numbers (reproduction gap), checks whether rankings survive a change of protocol, and calibrates 90 % intervals on the benchmark.

Compatibility patches are kept minimal and listed in `tables/b01_code_versions_and_patches.csv`. None changes a model, a loss or a metric.

In [ ]:
BENCH_CATS = {"NYC": ["Burglary", "Larceny", "Robbery", "Assault"], "CHI": ["Theft", "Battery", "Assault", "Damage"]}
BENCH_CITIES = ["NYC", "CHI"]
# Table III of Li et al. (2022, ICDE), ST-HSL paper: MAE, MAPE per category. Verify against the paper before quoting.
PUBLISHED = {
    "NYC": {"ARIMA": [0.8999, 0.6305, 1.3015, 0.6268, 0.9558, 0.5969, 0.9983, 0.6198], "SVM": [1.1604, 0.7653, 1.4979, 0.6417, 1.1278, 0.6733, 1.1928, 0.6964],
            "ST-ResNet": [0.8680, 0.5603, 1.1082, 0.5329, 0.8717, 0.5209, 0.9645, 0.5749], "DCRNN": [0.8176, 0.5324, 1.0732, 0.5492, 0.9189, 0.5532, 0.9692, 0.5955],
            "STGCN": [0.8366, 0.5404, 1.0629, 0.5295, 0.9035, 0.5441, 0.9375, 0.5757], "GWN": [0.7993, 0.5235, 1.0493, 0.5405, 0.8681, 0.5351, 0.8866, 0.5646],
            "STtrans": [0.8617, 0.5592, 1.0896, 0.5478, 0.8839, 0.5651, 0.9363, 0.5679], "DeepCrime": [0.8227, 0.5508, 1.0618, 0.5351, 0.8841, 0.5537, 0.9222, 0.5677],
            "MTGNN": [0.8329, 0.5439, 1.0473, 0.5330, 0.8759, 0.5457, 0.9090, 0.5714], "ST-SHN": [0.8012, 0.5198, 1.0431, 0.5291, 0.8717, 0.5362, 0.9169, 0.5682],
            "DMSTGCN": [0.8376, 0.5485, 1.0410, 0.5464, 0.8597, 0.5403, 0.9036, 0.5601], "ST-HSL": [0.7329, 0.4788, 1.0316, 0.5040, 0.7912, 0.4595, 0.8484, 0.5029]},
    "CHI": {"ARIMA": [1.5965, 0.5720, 1.3212, 0.5792, 0.8691, 0.6044, 1.0430, 0.6134], "SVM": [1.7711, 0.5629, 1.3493, 0.6027, 1.0879, 0.6560, 1.1313, 0.5721],
            "ST-ResNet": [1.3931, 0.5488, 1.1519, 0.5719, 0.7679, 0.4633, 0.9064, 0.5018], "DCRNN": [1.3699, 0.5770, 1.1583, 0.5528, 0.7639, 0.4600, 0.8764, 0.4756],
            "STGCN": [1.3628, 0.5359, 1.1512, 0.5761, 0.7963, 0.4810, 0.9068, 0.4959], "GWN": [1.3211, 0.5502, 1.1331, 0.5503, 0.7493, 0.4580, 0.8584, 0.4850],
            "STtrans": [1.3404, 0.5356, 1.1466, 0.5684, 0.7671, 0.4499, 0.8987, 0.4842], "DeepCrime": [1.3391, 0.5430, 1.1290, 0.5389, 0.7737, 0.4616, 0.9096, 0.4960],
            "ST-MetaNet": [1.3369, 0.5369, 1.1762, 0.5748, 0.7904, 0.4753, 0.8907, 0.4756], "MTGNN": [1.3054, 0.5378, 1.1307, 0.5597, 0.7571, 0.4572, 0.8667, 0.4859],
            "ST-SHN": [1.3231, 0.5310, 1.1348, 0.5544, 0.7758, 0.4574, 0.8741, 0.4747], "DMSTGCN": [1.3292, 0.5291, 1.1297, 0.5552, 0.8058, 0.4759, 0.8698, 0.4877],
            "ST-HSL": [1.2952, 0.4929, 1.1016, 0.5231, 0.6665, 0.3996, 0.8446, 0.4644]}}
REPO_URLS = {"STHSL": "https://github.com/LZH-YS1998/STHSL", "ST-SHN": "https://github.com/akaxlh/ST-SHN",
             "Graph-WaveNet": "https://github.com/nnzhan/Graph-WaveNet", "MTGNN": "https://github.com/nnzhan/MTGNN"}
PATCHES = {"STHSL": "engine.py: running loss stored with .item() (memory only); train.py: device and seed read from arguments/environment (official default seed 523 = our seed 0)",
           "ST-SHN": "TensorFlow 1 code run with tf.compat.v1; xavier_initializer → glorot_uniform (same initialiser); softmax on 6-D tensor written out (numerically identical); seed from environment; predictions and validation metrics logged every epoch",
           "Graph-WaveNet": "nn.Conv1d with 2-D kernels replaced by nn.Conv2d (old PyTorch accepted this silently; identical computation)",
           "MTGNN": "none"}

def git_clone(name):
    dest = REPOS / name
    if not dest.exists():
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URLS[name], str(dest)], capture_output=True, text=True)
        if r.returncode != 0: raise RuntimeError(f"could not clone {name}: {r.stderr[-300:]}")
    h = subprocess.run(["git", "rev-parse", "HEAD"], cwd=dest, capture_output=True, text=True).stdout.strip()
    return dest, h

def md5(p): return hashlib.md5(open(p, "rb").read()).hexdigest()
def bench_load(c):
    import pickle
    t = {s: np.asarray(pickle.load(open(REPOS / "STHSL" / "Datasets" / f"{c}_crime" / f"{s}.pkl", "rb")), np.float32) for s in ("trn", "val", "tst")}
    return np.concatenate([t["trn"], t["val"], t["tst"]], 2), t["trn"].shape[2], t["val"].shape[2], t["tst"].shape[2]

from pathlib import Path
def patch_stshn(root):
    """Minimal patches so the official TF1 ST-SHN code runs on TF2 (compat.v1), plus logging hooks.
    Model, loss, data handling and the official metrics are unchanged."""
    root = Path(root)
    nn_ = root / "Utils" / "NNLayers.py"; s = nn_.read_text()
    if "compat.v1" not in s:
        s = s.replace("import tensorflow as tf\nfrom tensorflow.contrib.layers import xavier_initializer",
                      "import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()\nxavier_initializer = tf.glorot_uniform_initializer", 1)
        assert "compat.v1" in s, "NNLayers import pattern not found"; nn_.write_text(s)
    m = root / "HG_ST_labcode.py"; s = m.read_text()
    if "compat.v1" in s: return
    rep = [
        ("import tensorflow as tf\n", "import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()\nimport os as _os, json as _json\n"
         "def _softmax(x, axis):\n\tx = x - tf.reduce_max(x, axis=axis, keepdims=True); e = tf.exp(x)\n\treturn e / tf.reduce_sum(e, axis=axis, keepdims=True)\n"),
        # hook 1: keep the per-cell predictions of every evaluation pass (aligned with the day index)
        ("\t\tnum = len(ids)\n\n\t\tsteps = int(np.ceil(num / args.batch))\n\t\tfor i in range(steps):\n\t\t\tst = i * args.batch\n\t\t\ted = min((i+1) * args.batch, num)\n\t\t\tbatIds = ids[st: ed]\n\n\t\t\ttem = self.sampTestBatch(batIds, tstTensor, inpTensor)",
         "\t\tnum = len(ids)\n\t\tself._P = np.zeros([args.areaNum, tstTensor.shape[1], args.offNum], np.float32)\n\n\t\tsteps = int(np.ceil(num / args.batch))\n\t\tfor i in range(steps):\n\t\t\tst = i * args.batch\n\t\t\ted = min((i+1) * args.batch, num)\n\t\t\tbatIds = ids[st: ed]\n\n\t\t\ttem = self.sampTestBatch(batIds, tstTensor, inpTensor)"),
        ("\t\t\t\tpreds, preLoss, regLoss, loss, sqLoss, absLoss, tstNums, apeLoss, posNums = res\n",
         "\t\t\t\tpreds, preLoss, regLoss, loss, sqLoss, absLoss, tstNums, apeLoss, posNums = res\n\t\t\t\tself._P[:, batIds[0], :] = preds\n"),
        # hook 2: after every epoch evaluate the validation block too, and store validation metrics + test predictions
        ("\t\t\tif test:\n\t\t\t\treses = self.testEpoch(self.handler.tstT, np.concatenate([self.handler.trnT, self.handler.valT], axis=1))\n",
         "\t\t\tif test:\n\t\t\t\tvres = self.testEpoch(self.handler.valT, self.handler.trnT)\n\t\t\t\treses = self.testEpoch(self.handler.tstT, np.concatenate([self.handler.trnT, self.handler.valT], axis=1))\n\t\t\t\tself._hist.append(dict(epoch=ep, val=vres, test=reses)); self._preds.append(self._P.copy())\n\t\t\t\tnp.save(_os.environ.get('STSHN_OUT', 'stshn') + '_preds.npy', np.stack(self._preds)); _json.dump([{k: {kk: float(vv) for kk, vv in v.items()} if isinstance(v, dict) else v for k, v in h.items()} for h in self._hist], open(_os.environ.get('STSHN_OUT', 'stshn') + '_hist.json', 'w'))\n"),
        ("\t\tbestRes = None\n\t\tfor ep in range(stloc, args.epoch):", "\t\tbestRes = None\n\t\tself._hist, self._preds = [], []\n\t\tfor ep in range(stloc, args.epoch):"),
        # seed from the environment (the official code sets none)
        ("\tconfig = tf.ConfigProto()\n", "\t_sd = int(_os.environ.get('STSHN_SEED', '0')); np.random.seed(_sd); tf.set_random_seed(_sd)\n\tconfig = tf.ConfigProto()\n"),
    ]
    for a, b in rep:
        assert a in s, "ST-SHN patch pattern not found: " + a[:60]
        s = s.replace(a, b, 1)
    assert s.count("att = tf.nn.softmax(") == 2, "ST-SHN softmax pattern not found"
    s = s.replace("att = tf.nn.softmax(", "att = _softmax(")    # TF2 kernels reject softmax on 6-D tensors; same softmax written out
    m.write_text(s)

with part("I.1 benchmark: code, data and protocol audit"):
    VERS = []
    for n in REPO_URLS:
        try: d, h = git_clone(n); VERS.append(dict(repository=n, url=REPO_URLS[n], commit=h, patches=PATCHES[n]))
        except Exception as e: VERS.append(dict(repository=n, url=REPO_URLS[n], commit="NOT AVAILABLE", patches=str(e)[:200]))
    VERS = save_table(pd.DataFrame(VERS), "b01_code_versions_and_patches"); display(VERS)
    BENCH_OK = (REPOS / "STHSL" / "Datasets").exists()
    # same data in both official repositories?
    same = []
    for c in BENCH_CITIES:
        for s in ("trn", "val", "tst"):
            a, b = REPOS / "STHSL" / "Datasets" / f"{c}_crime" / f"{s}.pkl", REPOS / "ST-SHN" / "Datasets" / f"{c}_crime" / f"{s}.pkl"
            same.append(dict(city=c, split=s, sthsl_md5=md5(a) if a.exists() else None, stshn_md5=md5(b) if b.exists() else None))
    same = pd.DataFrame(same); same["identical"] = same["sthsl_md5"] == same["stshn_md5"]; save_table(same, "b02_dataset_identity"); display(same)

    # automatic audit of the evaluation code (run on the untouched clones, before any patch)
    def code_has(repo, file, pattern):
        p = REPOS / repo / file
        return bool(p.exists() and re.search(pattern, p.read_text()))
    AUDIT = [
        ("ST-HSL", "scored cell-days", "only cell-days with at least one crime (mask = label > 0)", code_has("STHSL", "engine.py", r"mask = 1 \* \(label > 0\)")),
        ("ST-HSL", "training loss", "squared error on every cell-day (mask = label >= 0)", code_has("STHSL", "engine.py", r"retLabels = \(label >= 0\) \* 1")),
        ("ST-HSL", "reported epoch", "best validation RMSE + MAE", code_has("STHSL", "train.py", r"val_metrics = res_eval\['RMSE'\] \+ res_eval\['MAE'\]")),
        ("ST-HSL", "random seed", "fixed (seed_torch, 523)", code_has("STHSL", "utils.py", r"def seed_torch\(seed=523\)")),
        ("ST-SHN", "scored cell-days", "crime days only, and only cells with crime on 20–80 % of training days",
         code_has("ST-SHN", "HG_ST_labcode.py", r"mask = self\.handler\.tstLocs \* \(label > 0\)") and code_has("ST-SHN", "DataHandler.py", r"\(percent > 0\.2\) \* \(percent < 0\.8\)")),
        ("ST-SHN", "training loss", "squared error on crime days only (regression: negRate = 0)", code_has("ST-SHN", "HG_ST_labcode.py", r"negRate = 0")),
        ("ST-SHN", "reported epoch", "epoch with the best TEST MAPE (the test set chooses the epoch)",
         code_has("ST-SHN", "HG_ST_labcode.py", r"reses = self\.testEpoch\(self\.handler\.tstT.*\n.*bestRes\['MAPE'\] > reses\['MAPE'\]")),
        ("ST-SHN", "random seed", "none set", not code_has("ST-SHN", "HG_ST_labcode.py", r"set_random_seed|np\.random\.seed")),
    ]
    AUDIT = save_table(pd.DataFrame(AUDIT, columns=["code", "aspect", "what the official code does", "verified in code"]), "b03_protocol_audit"); display(AUDIT)

    # how much of the test set does the official metric ignore?
    BSTAT = []
    for c in BENCH_CITIES:
        Xall, ntr, nva, nte = bench_load(c); R, Cc, T, K = Xall.shape; tst = Xall[:, :, ntr + nva:].reshape(R * Cc, nte, K)
        act = (Xall[:, :, :ntr] > 0).mean(2).reshape(R * Cc, K); tl = (act > 0.2) & (act < 0.8)
        for k, n in enumerate(BENCH_CATS[c]):
            y = tst[:, :, k]
            BSTAT.append(dict(city=c, category=n, cells=R * Cc, test_days=nte, mean_daily_count=y.mean(), share_zero_cell_days=(y == 0).mean(),
                              share_scored_P1=(y > 0).mean(), cells_in_P2_subset=int(tl[:, k].sum()), share_scored_P2=((y > 0) & tl[:, k][:, None]).mean()))
    BSTAT = save_table(pd.DataFrame(BSTAT), "b04_benchmark_sparsity"); display(BSTAT.round(3))

### Benchmark models (defined here, run at the end of the notebook)
**Ours**: the global gradient-boosted Poisson model (LightGBM and XGBoost; the one with the lower validation deviance is reported) with lags 1–7, 7/14/30-day means, the same weekday over 4 weeks, the 8-neighbour sums, the cell total and the city total of yesterday, trained on the official training days and early-stopped on the 30 validation days. **Baselines**: last day, 30-day mean, a constant 1 and the protocol-aware zero-truncated Poisson median of our forecast (the best guess *given that a crime happens*, which the official metric rewards). **Re-runs**: ST-HSL (official code, 25 epochs, epoch chosen on validation as in its code), ST-SHN (official code, regression task, 10 epochs as in its defaults; reported both as its code reports – best *test* epoch – and with the epoch chosen on validation), Graph WaveNet and MTGNN (official model code, our training loop with the ST-HSL rule: z-scored inputs, 30-day window, squared-error loss on all cell-days, epoch chosen on validation RMSE + MAE).

In [ ]:
def bench_features(X, days):
    """X: (rows, cols, days, K). One row per (day, cell, category) in the order used by the official code."""
    R, Cc, T, K = X.shape; d = np.asarray(days)
    cs = np.concatenate([np.zeros((R, Cc, 1, K), np.float32), np.cumsum(X, 2)], 2)
    Pd = np.pad(X, ((1, 1), (1, 1), (0, 0), (0, 0))); NB = sum(Pd[1 + dy:1 + dy + R, 1 + dx:1 + dx + Cc] for dy in (-1, 0, 1) for dx in (-1, 0, 1)) - X
    csn = np.concatenate([np.zeros((R, Cc, 1, K), np.float32), np.cumsum(NB, 2)], 2)
    tot = X.sum(3, keepdims=True); cst = np.concatenate([np.zeros((R, Cc, 1, 1), np.float32), np.cumsum(tot, 2)], 2)
    f = {f"lag{l}": X[:, :, d - l] for l in range(1, 8)}
    for w in (7, 14, 30): f[f"m{w}"] = (cs[:, :, d] - cs[:, :, d - w]) / w
    f["same_dow4"] = np.mean([X[:, :, d - 7 * j] for j in (1, 2, 3, 4)], 0)
    f["nb1"] = NB[:, :, d - 1]; f["nb7"] = (csn[:, :, d] - csn[:, :, d - 7]) / 7; f["nb30"] = (csn[:, :, d] - csn[:, :, d - 30]) / 30
    f["cell_all7"] = np.broadcast_to((cst[:, :, d] - cst[:, :, d - 7]) / 7, f["lag1"].shape)
    f["city_lag1"] = np.broadcast_to(X.sum((0, 1), keepdims=True)[:, :, d - 1], f["lag1"].shape)
    df = pd.DataFrame({k: np.ascontiguousarray(np.moveaxis(v, 2, 0)).reshape(-1) for k, v in f.items()})
    n = len(d); df["cell"] = pd.Categorical(np.tile(np.repeat(np.arange(R * Cc), K), n)); df["cat"] = pd.Categorical(np.tile(np.arange(K), n * R * Cc))
    df["dow"] = np.repeat(d % 7, R * Cc * K)
    return df, np.moveaxis(X[:, :, d], 2, 0).reshape(-1)
def ztp_median(mu):
    mu = np.maximum(mu, 1e-3); p0 = np.exp(-mu); out = np.ones_like(mu)
    for k in range(1, 80): out = np.where((_pois.cdf(k, mu) - p0) / (1 - p0) < 0.5, k + 1, out)
    return out

# ---------------------------------------------------------------- ST-HSL (official code)
def prep_sthsl():
    d = REPOS / "STHSL"; eg = (d / "engine.py").read_text()
    if "epochLoss += loss\n" in eg: (d / "engine.py").write_text(eg.replace("epochLoss += loss\n", "epochLoss += loss.item()\n", 1))
    tr = (d / "train.py").read_text()
    if "torch.device(args.device)" not in tr: tr = tr.replace("torch.device('cuda:0')", "torch.device(args.device)")
    if "STHSL_SEED" not in tr:
        assert "    seed_torch()\n" in tr, "ST-HSL seed pattern not found"
        tr = tr.replace("    seed_torch()\n", "    import os as _os; seed_torch(int(_os.environ.get('STHSL_SEED', '523')))\n", 1)
    (d / "train.py").write_text(tr)
    (d / "dump_preds.py").write_text('''
import numpy as np, torch
from Params import args
from DataHandler import DataHandler
from model import STHSL
import engine
handler = DataHandler(); model = STHSL().to(args.device)
model.load_state_dict(torch.load(args.checkpoint, map_location=args.device)); model.eval()
def run(T, inp):
    ids = np.arange(T.shape[1]); P = []
    with torch.no_grad():
        for st in range(0, len(ids), args.batch):
            b = ids[st:st + args.batch]
            feats, labels, mask = engine.sampleTestBatch(b, st, st + len(b), T, inp, handler)
            feats = torch.Tensor(feats).to(args.device)
            out = model(feats, feats[:, np.random.permutation(args.areaNum)])[-1]
            P.append(handler.zInverse(out).cpu().numpy())
    return np.concatenate(P)
np.save("sthsl_pred_tst.npy", run(handler.tstT, np.concatenate([handler.trnT, handler.valT], axis=1)))
np.save("sthsl_pred_val.npy", run(handler.valT, handler.trnT))
''')

def run_logged(cmd, cwd, logfile, timeout_s, env=None):
    with open(logfile, "w") as lf:
        p = subprocess.Popen(cmd, cwd=cwd, stdout=lf, stderr=subprocess.STDOUT, env={**os.environ, **(env or {})})
        try: rc = p.wait(timeout=timeout_s)
        except subprocess.TimeoutExpired: p.kill(); rc = "timeout"
    return rc

def run_sthsl(c, seed, epochs, timeout_s):
    d = REPOS / "STHSL"; (d / "Save" / c).mkdir(parents=True, exist_ok=True)
    for f in (d / "Save" / c).glob("*.pth"): f.unlink()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    dev = "cuda:0" if DEVICE.type == "cuda" else "cpu"; t = time.time(); lf = LOGS / f"sthsl_{c}_seed{seed}.log"
    rc = run_logged([sys.executable, "train.py", "--data", c, "--device", dev, "--epoch", str(epochs), "--save", "./Save/"], d, lf, timeout_s,
                    env={"STHSL_SEED": str(523 if seed == 0 else seed)})
    txt = lf.read_text(errors="ignore").replace("\r", "\n"); ep, best_ep, n_ep = 0, None, 0
    for line in txt.splitlines():
        mm = re.match(r"Epoch\s+(\d+) Training Time", line.strip())
        if mm: ep = int(mm.group(1)); n_ep = ep
        if "Val metrics decrease" in line: best_ep = ep
    if best_ep is None: log(f"  ST-HSL {c} seed {seed}: no finished epoch (exit {rc}); see {lf.name}"); return None
    ck = sorted((d / "Save" / c).glob(f"_epoch_{best_ep}_MAE_*.pth"))
    if not ck: log(f"  ST-HSL {c} seed {seed}: checkpoint of epoch {best_ep} missing"); return None
    rc2 = run_logged([sys.executable, "dump_preds.py", "--data", c, "--device", dev, "--checkpoint", str(ck[0])], d, LOGS / f"sthsl_{c}_seed{seed}_dump.log", 1800)
    if rc2 != 0: log(f"  ST-HSL {c} seed {seed}: prediction dump failed ({rc2})"); return None
    info = dict(model="ST-HSL", city=c, seed=seed, epochs_run=n_ep, epochs_planned=epochs, selected_epoch=best_ep, selection="validation (official rule)",
                minutes=(time.time() - t) / 60, finished=(rc == 0))
    log(f"  ST-HSL {c} seed {seed}: {n_ep}/{epochs} epochs, validation-best epoch {best_ep}, {info['minutes']:.0f} min")
    # (days, cells, K) predictions
    return np.load(d / "sthsl_pred_tst.npy"), np.load(d / "sthsl_pred_val.npy"), info

# ---------------------------------------------------------------- ST-SHN (official TensorFlow 1 code)
def prep_stshn():
    src, dst = REPOS / "ST-SHN", REPOS / "ST-SHN-run"
    if not dst.exists(): shutil.copytree(src, dst)
    patch_stshn(dst); (dst / "History").mkdir(exist_ok=True); (dst / "Models").mkdir(exist_ok=True)
    return dst
def run_stshn(c, seed, timeout_s, epochs=None):
    d = REPOS / "ST-SHN-run"; t = time.time(); lf = LOGS / f"stshn_{c}_seed{seed}.log"; outp = str(d / f"out_{c}_s{seed}")
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    cmd = [sys.executable, "HG_ST_labcode.py", "--data", c, "--task", "r", "--save_path", f"{c}_s{seed}"] + (["--epoch", str(epochs)] if epochs else [])
    env = {"STSHN_OUT": outp, "STSHN_SEED": str(seed), "TF_CPP_MIN_LOG_LEVEL": "2"}
    if DEVICE.type != "cuda": env["CUDA_VISIBLE_DEVICES"] = ""
    rc = run_logged(cmd, d, lf, timeout_s, env)
    if not Path(outp + "_preds.npy").exists(): log(f"  ST-SHN {c} seed {seed}: no epoch finished (exit {rc}); see {lf.name}"); return None
    P = np.load(outp + "_preds.npy"); H = json.load(open(outp + "_hist.json"))            # P: (epochs, cells, days, K)
    vm = np.array([h["val"]["MAPE"] for h in H]); tm = np.array([h["test"]["MAPE"] for h in H])
    e_val, e_tst = int(np.argmin(vm)), int(np.argmin(tm))
    info = dict(model="ST-SHN", city=c, seed=seed, epochs_run=len(H), epochs_planned=epochs or 10, selected_epoch_validation=e_val + 1,
                selected_epoch_test=e_tst + 1, minutes=(time.time() - t) / 60, finished=(rc == 0))
    log(f"  ST-SHN {c} seed {seed}: {len(H)} epochs, epoch by validation {e_val + 1}, by test (official) {e_tst + 1}, {info['minutes']:.0f} min")
    return np.transpose(P[e_val], (1, 0, 2)), np.transpose(P[e_tst], (1, 0, 2)), info

# ---------------------------------------------------------------- Graph WaveNet / MTGNN (official model code, ST-HSL training rule)
def load_module(path, name, extra_path=None):
    import importlib.util
    if extra_path and str(extra_path) not in sys.path: sys.path.insert(0, str(extra_path))
    spec = importlib.util.spec_from_file_location(name, path); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
def grid_adj(R, Cc):
    N = R * Cc; A = np.eye(N, dtype=np.float32)
    for i in range(R):
        for j in range(Cc):
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if 0 <= i + di < R and 0 <= j + dj < Cc: A[i * Cc + j, (i + di) * Cc + (j + dj)] = 1
    return A
def make_graph_model(kind, N, K, R, Cc):
    if kind == "Graph WaveNet":
        gd = REPOS / "Graph-WaveNet"; mp = gd / "model_patched.py"
        if not mp.exists(): mp.write_text((gd / "model.py").read_text().replace("nn.Conv1d(", "nn.Conv2d("))
        gw = load_module(mp, "gwn_model_patched")
        A = grid_adj(R, Cc); P = A / A.sum(1, keepdims=True)
        sup = [torch.tensor(P, device=DEVICE), torch.tensor(P.T.copy(), device=DEVICE)]
        return gw.gwnet(DEVICE, N, dropout=0.3, supports=sup, gcn_bool=True, addaptadj=True, aptinit=None, in_dim=K, out_dim=K,
                        residual_channels=32, dilation_channels=32, skip_channels=256, end_channels=512).to(DEVICE)
    mt = load_module(REPOS / "MTGNN" / "net.py", "mtgnn_net", REPOS / "MTGNN")
    return mt.gtnet(True, True, 2, N, DEVICE, predefined_A=None, static_feat=None, dropout=0.3, subgraph_size=min(20, N), node_dim=40,
                    dilation_exponential=1, conv_channels=32, residual_channels=32, skip_channels=64, end_channels=128, seq_length=30,
                    in_dim=K, out_dim=K, layers=3, propalpha=0.05, tanhalpha=3, layer_norm_affline=True).to(DEVICE)
def train_graph_model(kind, c, seed, max_epochs=2 if FAST else 100, patience=10, batch=16, L=30):
    set_seed(seed); Xall, ntr, nva, nte = bench_load(c); R, Cc, T, K = Xall.shape; N = R * Cc
    S = Xall.reshape(N, T, K); mean, std = float(Xall[:, :, :ntr].mean()), float(Xall[:, :, :ntr].std())
    Z = torch.tensor((S - mean) / std, device=DEVICE); Y = torch.tensor(S, device=DEVICE)
    model = make_graph_model(kind, N, K, R, Cc); opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    back = torch.arange(L, 0, -1, device=DEVICE)
    def fwd(days):
        win = days[:, None] - back[None]                                        # (B, L): days d-L .. d-1
        x = Z[:, win, :].permute(1, 3, 0, 2)                                    # (B, K, N, L)
        out = model(x)[..., -1]                                                 # (B, K, N)
        return out.permute(0, 2, 1) * std + mean                                # (B, N, K) counts
    def predict(days):
        model.eval(); out = []
        with torch.no_grad():
            for s in range(0, len(days), 32): out.append(fwd(days[s:s + 32]).cpu().numpy())
        return np.concatenate(out)
    trd = torch.arange(L, ntr, device=DEVICE); vad = torch.arange(ntr, ntr + nva, device=DEVICE); ted = torch.arange(ntr + nva, T, device=DEVICE)
    yv = S[:, ntr:ntr + nva].transpose(1, 0, 2); best, best_state, bad, t = np.inf, None, 0, time.time()
    for ep in range(max_epochs):
        model.train(); perm = trd[torch.randperm(len(trd), device=DEVICE)]
        for s in range(0, len(perm), batch):
            b = perm[s:s + batch]; loss = F.mse_loss(fwd(b), Y[:, b, :].permute(1, 0, 2))
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        pv = predict(vad); pos = yv > 0; e = pv - yv
        crit = np.sqrt((e[pos] ** 2).mean()) + np.abs(e[pos]).mean()           # ST-HSL rule: validation RMSE + MAE on crime days
        if crit < best - 1e-5: best, bad, best_state, best_ep = crit, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}, ep + 1
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    info = dict(model=kind, city=c, seed=seed, epochs_run=ep + 1, epochs_planned=max_epochs, selected_epoch=best_ep, selection="validation (ST-HSL rule)",
                minutes=(time.time() - t) / 60, finished=True)
    log(f"  {kind} {c} seed {seed}: {ep + 1} epochs, best {best_ep}, {info['minutes']:.1f} min")
    return predict(ted), predict(vad), info

# ---------------------------------------------------------------- metrics under the three protocols
def p_metrics(p, y, cat, K, keep):
    """MAE / MAPE per category on the cell-days in `keep` (the official codes use keep = y > 0, ST-SHN also restricts cells)."""
    out = {}
    for k in range(K):
        m = keep & (cat == k); out[f"MAE_{k}"] = np.abs(p - y)[m].mean(); out[f"MAPE_{k}"] = (np.abs(p - y) / np.maximum(y, 1e-8))[m].mean()
    out["MAE"] = np.abs(p - y)[keep].mean(); out["MAPE"] = (np.abs(p - y) / np.maximum(y, 1e-8))[keep].mean(); return out
def full_metrics(p, y, cat, K):
    out = dict(MAE_all=np.abs(p - y).mean(), RMSE_all=np.sqrt(((p - y) ** 2).mean()), deviance_all=pdev(y, np.maximum(p, 0.01)),
               MAE_zero_days=np.abs(p - y)[y == 0].mean(), MAE_crime_days=np.abs(p - y)[y > 0].mean())
    for k in range(K): out[f"deviance_{k}"] = pdev(y[cat == k], np.maximum(p[cat == k], 0.01)); out[f"MAE_all_{k}"] = np.abs(p - y)[cat == k].mean()
    return out

## Part II. Weekly area forecasting in three cities
### II.1 Data
Open data downloaded live (no Kaggle dataset needed), year by year with retries; a finished download is reused if the notebook runs again in the same session.
- **Chicago**: *Crimes – 2001 to Present* (`ijzp-q8t2`), 77 community areas.
- **New York**: *NYPD Complaint Data Historic* (`qgea-i56i`) + *Current Year To Date* (`5uac-w243`), 77 precincts (116 merged into 105, from which it was carved in 2024).
- **San Francisco**: *Police Department Incident Reports: 2018 to Present* (`wg3w-h783`), 41 analysis neighbourhoods; one record per incident and category (rows of the same incident and category are de-duplicated).

All three use the same 11 crime types as the capstone (keyword mapping, saved to `tables/c01_offence_mapping.csv`). The panel is **city × area × type × week** (weeks start on Monday). Incomplete first and last weeks are dropped.

In [ ]:
SOURCES = {
    "CHI": dict(api="https://data.cityofchicago.org/resource/ijzp-q8t2.csv", date="date", first_year=2001,
                cols="date,primary_type,community_area,latitude,longitude"),
    "NYC": dict(api="https://data.cityofnewyork.us/resource/qgea-i56i.csv", date="cmplnt_fr_dt", first_year=2006,
                cols="cmplnt_fr_dt,ofns_desc,addr_pct_cd,latitude,longitude"),
    "NYC_YTD": dict(api="https://data.cityofnewyork.us/resource/5uac-w243.csv", date="cmplnt_fr_dt", first_year=2006,
                    cols="cmplnt_fr_dt,ofns_desc,addr_pct_cd,latitude,longitude"),
    "SF": dict(api="https://data.sfgov.org/resource/wg3w-h783.csv", date="incident_datetime", first_year=2018,
               cols="incident_datetime,incident_id,incident_category,analysis_neighborhood,latitude,longitude"),
}
CITY_CSV = {c: os.environ.get(f"OA_{c}_CSV", "") for c in CITIES}       # optional: your own extract with the API column names

def soda_download(key, dest, start_year, end_year=None, page=500_000, select=True):
    import urllib.request, urllib.parse, urllib.error
    src = SOURCES[key]; tmp = Path(str(dest) + ".part"); total, wrote_header = 0, False
    end_year = end_year or pd.Timestamp.today().year
    with open(tmp, "wb") as out:
        for yr in range(max(start_year, src["first_year"]), end_year + 1):
            offset, size, n_year = 0, page, 0
            while True:
                q = ({"$select": src["cols"]} if select else {}) | {"$where": f"{src['date']} between '{yr}-01-01T00:00:00' and '{yr}-12-31T23:59:59'",
                     "$order": ":id", "$limit": size, "$offset": offset}
                url = src["api"] + "?" + urllib.parse.urlencode(q); data, last = None, None
                for attempt in range(5):
                    try:
                        with urllib.request.urlopen(url, timeout=900) as r: data = r.read()
                        break
                    except urllib.error.HTTPError as e:
                        last = e
                        if e.code == 400: break                       # bad query (e.g. a renamed column): no point retrying
                        time.sleep(15 * (attempt + 1))
                    except Exception as e:
                        last = e; time.sleep(15 * (attempt + 1))
                if data is None: raise RuntimeError(f"{key}: request failed for {yr}: {last!r}")
                head, _, body = data.partition(b"\n"); n = body.count(b"\n") + (0 if (body.endswith(b"\n") or not body) else 1)
                if not wrote_header: out.write(head.strip() + b"\n"); wrote_header = True
                if body: out.write(body if body.endswith(b"\n") else body + b"\n")
                n_year += n; offset += n
                if n == 0: break
                if n < size:
                    if n in (1000, 50000): size = n; continue
                    break
            log(f"  {key} {yr}: {n_year:,} rows"); total += n_year
    if total == 0: raise RuntimeError(f"{key}: the portal returned no rows")
    tmp.replace(dest); return total
def get_file(key):
    if CITY_CSV.get(key) and Path(CITY_CSV[key]).exists(): return Path(CITY_CSV[key])
    dest = DATA / f"{key.lower()}_{START_YEAR}.csv"
    if not dest.exists():
        log(f"{key}: downloading from {START_YEAR} ...")
        try: soda_download(key, dest, START_YEAR)
        except RuntimeError as e:
            if "400" not in str(e): raise
            log(f"{key}: column selection refused by the portal ({str(e)[:120]}); downloading all columns instead"); soda_download(key, dest, START_YEAR, select=False)
    return dest

FINE = ["THEFT", "BURGLARY", "VEHICLE_THEFT", "DAMAGE", "FRAUD", "ASSAULT", "ROBBERY", "SEX_OFFENSE", "DRUGS", "WEAPONS", "OTHER"]
def map_offence(s):
    s = str(s).upper().strip()
    if "MOTOR VEHICLE" in s and ("THEFT" in s or "LARCENY" in s): return "VEHICLE_THEFT"
    if "ROBBERY" in s: return "ROBBERY"
    if "BURGLAR" in s: return "BURGLARY"
    if "SEX" in s or "RAPE" in s: return "SEX_OFFENSE"
    if "FRAUD" in s or "FORGERY" in s or "DECEPTIVE" in s or "EMBEZZLEMENT" in s: return "FRAUD"
    if any(k in s for k in ("ASSAULT", "BATTERY", "HOMICIDE", "MURDER", "KIDNAP", "MANSLAUGHTER")): return "ASSAULT"
    if "LARCENY" in s or s == "THEFT" or s.startswith("THEFT OF"): return "THEFT"
    if "DAMAGE" in s or "MISCHIEF" in s or "ARSON" in s or "VANDAL" in s: return "DAMAGE"
    if "NARCOTIC" in s or "DRUG" in s or "CANNABIS" in s: return "DRUGS"
    if "WEAPON" in s or "CONCEALED CARRY" in s: return "WEAPONS"
    return "OTHER"
BOX = {"CHI": (41.60, 42.10, -87.95, -87.50), "NYC": (40.45, 40.95, -74.30, -73.65), "SF": (37.65, 37.85, -122.55, -122.33)}

def load_city(c):
    t = time.time()
    if c == "CHI":
        df = pd.read_csv(get_file("CHI"), dtype=str)
        ts = pd.to_datetime(df["date"].str.slice(0, 19), format="%Y-%m-%dT%H:%M:%S", errors="coerce")
        bad = ts.isna()
        if bad.any(): ts[bad] = pd.to_datetime(df.loc[bad, "date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
        area = pd.to_numeric(df["community_area"], errors="coerce"); area = area.where((area >= 1) & (area <= 77))
        area = area.astype("Int64").astype(str).where(area.notna()); off = df["primary_type"]
    elif c == "NYC":
        frames = [pd.read_csv(get_file("NYC"), dtype=str)]
        try:
            ytd = DATA / "nyc_ytd.csv"
            if not ytd.exists() and not CITY_CSV.get("NYC"):
                yr = pd.Timestamp.today().year; soda_download("NYC_YTD", ytd, yr - 1, yr)
            if ytd.exists():
                y = pd.read_csv(ytd, dtype=str); d0 = pd.to_datetime(frames[0]["cmplnt_fr_dt"].str.slice(0, 10), errors="coerce").max()
                y = y[pd.to_datetime(y["cmplnt_fr_dt"].str.slice(0, 10), errors="coerce") > d0]; frames.append(y); log(f"  NYC year-to-date rows added: {len(y):,}")
        except Exception as e: log(f"  NYC year-to-date not used: {str(e)[:150]}")
        df = pd.concat(frames, ignore_index=True)
        d = df["cmplnt_fr_dt"].fillna("").str.slice(0, 10)
        d = d.where(~d.str.contains("/"), pd.to_datetime(d, format="%m/%d/%Y", errors="coerce").dt.strftime("%Y-%m-%d"))
        ts = pd.to_datetime(d, format="%Y-%m-%d", errors="coerce")
        area = pd.to_numeric(df["addr_pct_cd"], errors="coerce").replace({116: 105}); area = area.where((area >= 1) & (area <= 123))
        area = area.astype("Int64").astype(str).where(area.notna()); off = df["ofns_desc"]
    else:
        df = pd.read_csv(get_file("SF"), dtype=str)
        n0 = len(df); df = df.drop_duplicates(subset=["incident_id", "incident_category"]); log(f"  SF: {n0 - len(df):,} duplicate incident × category rows removed")
        ts = pd.to_datetime(df["incident_datetime"].str.slice(0, 19), format="%Y-%m-%dT%H:%M:%S", errors="coerce")
        bad = ts.isna()
        if bad.any(): ts[bad] = pd.to_datetime(df.loc[bad, "incident_datetime"], errors="coerce")
        area = df["analysis_neighborhood"].where(df["analysis_neighborhood"].notna() & (df["analysis_neighborhood"].str.strip() != "")); off = df["incident_category"]
    lat = pd.to_numeric(df["latitude"], errors="coerce"); lon = pd.to_numeric(df["longitude"], errors="coerce")
    b = BOX[c]; okc = lat.between(b[0], b[1]) & lon.between(b[2], b[3]); lat, lon = lat.where(okc), lon.where(okc)
    keep = ts.notna() & (ts.dt.year >= START_YEAR) & (ts <= pd.Timestamp.today()) & area.notna()
    ev = pd.DataFrame({"ts": ts, "area": area, "offence_raw": off.fillna("UNKNOWN").str.upper().str.strip(), "lat": lat.astype("float32"), "lon": lon.astype("float32")})[keep.values]
    log(f"{CITY_NAME[c]}: {len(df):,} rows read, {len(ev):,} kept ({time.time() - t:.0f}s)")
    return ev.reset_index(drop=True)

def area_sort_key(a): return (0, int(a)) if str(a).isdigit() else (1, str(a))
PANEL, GEO, MAPTAB = {}, {}, []
with part("II.1 city data and weekly panels"):
    for c in CITIES:
        try: ev = load_city(c)
        except Exception as e: log(f"{c} not available, continuing without it: {str(e)[:300]}"); continue
        om = {s: map_offence(s) for s in ev["offence_raw"].unique()}
        MAPTAB.append(ev.groupby("offence_raw").size().rename("n").reset_index().assign(city=c, type=lambda d: d["offence_raw"].map(om)))
        ev["type"] = ev["offence_raw"].map(om).map({f: i for i, f in enumerate(FINE)}).astype(np.int8)
        ev["date"] = ev["ts"].dt.normalize(); ev["week"] = ev["date"] - pd.to_timedelta(ev["date"].dt.dayofweek, unit="D")
        first = ev["week"].min() + pd.Timedelta(days=7); last = ev["week"].max() - pd.Timedelta(days=7 * DROP_LAST_WEEKS[c])
        ev = ev[(ev["week"] >= first) & (ev["week"] <= last)]
        weeks = pd.date_range(first, last, freq="7D"); areas = sorted(ev["area"].astype(str).unique(), key=area_sort_key)
        wi = ((ev["week"] - weeks[0]).dt.days // 7).values; ai = pd.Categorical(ev["area"].astype(str), categories=areas).codes
        C = np.zeros((len(weeks), len(areas), len(FINE)), np.int32); np.add.at(C, (wi, ai, ev["type"].values), 1)
        cent = ev[ev["lat"].notna()].groupby("area")[["lat", "lon"]].median().reindex(areas)
        PANEL[c] = dict(weeks=weeks, areas=areas, C=C, cent=cent.fillna(cent.mean()))
        g = ev[ev["lat"].notna() & (ev["date"] > ev["date"].max() - pd.Timedelta(days=730))]
        GEO[c] = g.sample(min(len(g), 300_000), random_state=SEED)[["lat", "lon", "area"]].reset_index(drop=True)   # for mapping census tracts to areas
        log(f"{CITY_NAME[c]}: {len(weeks)} weeks ({weeks[0].date()} – {weeks[-1].date()}) × {len(areas)} areas × {len(FINE)} types, {C.sum():,} incidents")
        del ev, g; gc.collect()
    CITIES_OK = [c for c in CITIES if c in PANEL]
    if MAPTAB: save_table(pd.concat(MAPTAB).sort_values(["city", "n"], ascending=[True, False]), "c01_offence_mapping")
    DS = save_table(pd.DataFrame([dict(city=c, first_week=PANEL[c]["weeks"][0].date(), last_week=PANEL[c]["weeks"][-1].date(), weeks=len(PANEL[c]["weeks"]),
                                       areas=len(PANEL[c]["areas"]), incidents=int(PANEL[c]["C"].sum()),
                                       mean_weekly_count_per_series=float(PANEL[c]["C"].mean()), share_zero_series_weeks=float((PANEL[c]["C"] == 0).mean()))
                                  for c in CITIES_OK]), "c02_panel_summary")
    display(DS)
    fig, ax = plt.subplots(figsize=(13, 4))
    for c in CITIES_OK: ax.plot(PANEL[c]["weeks"], PANEL[c]["C"].sum((1, 2)), lw=1, label=CITY_NAME[c])
    ax.axvspan(TRAIN_END, VAL_END, color="grey", alpha=.15, label="calibration block"); ax.axvline(VAL_END, color="k", ls=":")
    for y_ in ROLL_YEARS: ax.axvline(pd.Timestamp(f"{y_}-01-01"), color="tab:red", ls=":", lw=.6)
    ax.set_title("Weekly recorded incidents (red dotted = rolling-origin test years)"); ax.legend(); savefig("figC1_weekly_series")

### II.2 Ten forecasters, horizons 1–4 weeks, several seeds
Every **city × area × crime type** series is forecast *H* = 1, 2, 3 and 4 weeks ahead. Same split as the capstone: target weeks up to 2023 train (the last 26 of them for early stopping), January–June 2024 is the validation/calibration block, everything after is the test period.

| Forecaster | Family |
|---|---|
| Last week, seasonal naive (52 w), moving average (4 w), historical average (52 w), exponential smoothing (α = 0.3) | naive / statistical |
| Poisson GLM | count regression |
| LightGBM-Poisson, XGBoost-Poisson | global gradient-boosted trees |
| GRU (global), ST-GRU (with the neighbouring areas' series) | deep sequence / spatio-temporal |

Learned models are run with several seeds (trees: `SEEDS_TREE`; neural: `SEEDS_NN`, horizons `NN_HORIZONS`). The forecaster with the lowest validation deviance (mean over seeds, cities and horizons) is **selected before the test period is scored**. Significance: week-block bootstrap of the per-row deviance difference, separately for every seed.

In [ ]:
RATE_MODELS = {"lgbm": "LightGBM-Poisson", "xgb": "XGBoost-Poisson", "glm": "Poisson GLM", "gru": "GRU (global)", "stgru": "ST-GRU (graph diffusion input)",
               "ma4": "Moving average (4 w)", "ma52": "Historical average (52 w)", "ses": "Exp. smoothing (0.3)", "naive": "Last week", "snaive": "Seasonal naive (52 w)"}
LEARNED = ["lgbm", "xgb", "glm", "gru", "stgru"]; TREES = ["lgbm", "xgb"]; NEURAL = ["glm", "gru", "stgru"]; NAIVE = ["ma4", "ma52", "ses", "naive", "snaive"]

def series_tables(c):
    C = PANEL[c]["C"].astype(np.float32); W = C.shape[0]
    cs = np.concatenate([np.zeros((1,) + C.shape[1:], np.float32), np.cumsum(C, 0)])
    S = np.zeros_like(C); S[0] = C[0]
    for t in range(1, W): S[t] = 0.3 * C[t] + 0.7 * S[t - 1]
    return C, cs, S

def build_rows(c, H, origins):
    C, cs, S = SER[c]; W, A, K = C.shape; o = np.asarray(origins); n = len(o)
    def rm(w): return (cs[o + 1] - cs[np.maximum(o + 1 - w, 0)]) / w
    f = {f"lag{l}": C[o - l] for l in range(8)}
    f.update(rm4=rm(4), rm13=rm(13), rm52=rm(52))
    tgt = o + H
    f["ya"] = C[tgt - 52]; f["ya5"] = np.mean([C[np.clip(tgt - 52 + d, 0, W - 1)] for d in (-2, -1, 0, 1, 2)], 0)
    last4 = cs[o + 1] - cs[np.maximum(o - 3, 0)]
    f["area_tot4"] = np.broadcast_to(np.log1p(last4.sum(-1, keepdims=True)), (n, A, K))
    f["city_type4"] = np.broadcast_to(np.log1p(last4.sum(1, keepdims=True)), (n, A, K))
    weeks = PANEL[c]["weeks"]; tw = weeks[0] + pd.to_timedelta(7 * tgt, unit="D")
    woy = np.asarray(tw.isocalendar().week, dtype=np.float32)
    f["woy_sin"] = np.broadcast_to(np.sin(2 * np.pi * woy / 52.18)[:, None, None], (n, A, K))
    f["woy_cos"] = np.broadcast_to(np.cos(2 * np.pi * woy / 52.18)[:, None, None], (n, A, K))
    df = pd.DataFrame({k: np.asarray(v, np.float32).reshape(-1) for k, v in f.items()})
    df["area"] = pd.Categorical(np.tile(np.repeat([AREA_GLOBAL[f"{c}_{a}"] for a in PANEL[c]["areas"]], K), n), categories=range(len(AREA_GLOBAL)))
    df["type"] = pd.Categorical(np.tile(np.arange(K), n * A), categories=range(K))
    df["city"] = pd.Categorical([c] * len(df), categories=CITIES_OK)
    meta = pd.DataFrame({"city": c, "origin": np.repeat(o, A * K).astype(np.int32), "target_week": np.repeat(np.asarray(tw), A * K),
                         "area": np.tile(np.repeat(PANEL[c]["areas"], K), n), "type": np.tile(np.arange(K), n * A).astype(np.int8)})
    meta["ai"] = np.tile(np.repeat(np.arange(A), K), n).astype(np.int32)
    meta["y"] = C[tgt].reshape(-1)
    meta["naive"] = C[o].reshape(-1); meta["ma4"] = f["rm4"].reshape(-1); meta["ses"] = S[o].reshape(-1); meta["ma52"] = f["rm52"].reshape(-1)
    meta["snaive"] = np.asarray(f["ya"]).reshape(-1)
    return df, meta

# ---------------------------------------------------------------- Poisson GLM, GRU and ST-GRU (torch)
GLM_NUM = [f"lag{l}" for l in range(8)] + ["rm4", "rm13", "rm52", "ya", "ya5", "area_tot4", "city_type4", "woy_sin", "woy_cos"]
def glm_design(X):
    num = X[GLM_NUM].values.astype(np.float32); num[:, :13] = np.log1p(num[:, :13]); return num
class PoissonGLM(nn.Module):
    def __init__(self, n_num):
        super().__init__(); self.lin = nn.Linear(n_num, 1); self.ea = nn.Embedding(len(AREA_GLOBAL), 1); self.et = nn.Embedding(len(FINE), 1); self.ec = nn.Embedding(len(CITIES_OK), 1)
        for e in (self.ea, self.et, self.ec): nn.init.zeros_(e.weight)
    def forward(self, xn, a, t, c): return (self.lin(xn) + self.ea(a) + self.et(t) + self.ec(c)).squeeze(-1)
SEQ_L = 52
class SeqForecaster(nn.Module):
    def __init__(self, spatial, hid=64):
        super().__init__(); self.spatial = spatial
        self.gru = nn.GRU(4 if spatial else 3, hid, batch_first=True)
        self.ea = nn.Embedding(len(AREA_GLOBAL), 16); self.et = nn.Embedding(len(FINE), 8); self.ec = nn.Embedding(len(CITIES_OK), 4)
        self.head = nn.Sequential(nn.Linear(hid + 28 + 2, 64), nn.GELU(), nn.Linear(64, 1))
    def forward(self, seq, ag, t, c, woy):
        _, h = self.gru(seq); return self.head(torch.cat([h[-1], self.ea(ag), self.et(t), self.ec(c), woy], 1)).squeeze(-1)
def seq_batch(c, o, a, k, spatial):
    S_ = SEQ[c]; win = o[:, None] - torch.arange(SEQ_L - 1, -1, -1, device=DEVICE)[None]
    parts = [S_["x"][win, a[:, None], k[:, None]], S_["at"][win, a[:, None]], S_["ct"][win, k[:, None]]]
    if spatial: parts.append(S_["nb"][win, a[:, None], k[:, None]])
    return torch.stack(parts, -1)

def train_torch_rate(kind, X, M, fit, es, seed, epochs=3 if FAST else 20, batch=2048, lr=2e-3):
    set_seed(seed); ccode = {c: i for i, c in enumerate(CITIES_OK)}
    o = torch.as_tensor(M["origin"].values.astype(np.int64), device=DEVICE); ai = torch.as_tensor(M["ai"].values.astype(np.int64), device=DEVICE)
    ag = torch.as_tensor(X["area"].cat.codes.values.astype(np.int64), device=DEVICE); ty = torch.as_tensor(M["type"].values.astype(np.int64), device=DEVICE)
    cc = torch.as_tensor(M["city"].map(ccode).values.astype(np.int64), device=DEVICE)
    y = torch.as_tensor(M["y"].values.astype(np.float32), device=DEVICE)
    woy = torch.as_tensor(X[["woy_sin", "woy_cos"]].values.astype(np.float32), device=DEVICE)
    fit_idx = torch.as_tensor(np.where(fit)[0], device=DEVICE); es_idx = torch.as_tensor(np.where(es)[0], device=DEVICE)
    if kind == "glm":
        xn = torch.as_tensor(glm_design(X), device=DEVICE); mu_, sd_ = xn[fit_idx].mean(0), xn[fit_idx].std(0) + 1e-6
        xn = (xn - mu_) / sd_; model = PoissonGLM(xn.shape[1]).to(DEVICE)
        fwd = lambda idx: model(xn[idx], ag[idx], ty[idx], cc[idx])
    else:
        model = SeqForecaster(kind == "stgru").to(DEVICE)
        def fwd(idx):
            out = torch.empty(len(idx), device=DEVICE)
            for cn, ci in ccode.items():
                sel = (cc[idx] == ci)
                if sel.any():
                    j = idx[sel]; out[sel] = model(seq_batch(cn, o[j], ai[j], ty[j], kind == "stgru"), ag[j], ty[j], cc[j], woy[j])
            return out
    opt = torch.optim.AdamW(model.parameters(), lr=lr * (5 if kind == "glm" else 1), weight_decay=1e-4)
    def eval_dev(idx):
        model.eval(); tot = 0.0
        with torch.no_grad():
            for s in range(0, len(idx), 32768):
                b = idx[s:s + 32768]; tot += F.poisson_nll_loss(fwd(b), y[b], log_input=True, full=True, reduction="sum").item()
        return tot / len(idx)
    best, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        model.train(); perm = fit_idx[torch.randperm(len(fit_idx), device=DEVICE)]
        for s in range(0, len(perm), batch):
            b = perm[s:s + batch]; loss = F.poisson_nll_loss(fwd(b), y[b], log_input=True, full=False)
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        v = eval_dev(es_idx)
        if v < best - 1e-5: best, bad, best_state = v, 0, {k_: t_.detach().clone() for k_, t_ in model.state_dict().items()}
        else:
            bad += 1
            if bad >= 3: break
    model.load_state_dict(best_state); model.eval(); out = []; allidx = torch.arange(len(M), device=DEVICE)
    with torch.no_grad():
        for s in range(0, len(M), 32768): out.append(torch.exp(fwd(allidx[s:s + 32768]).clamp(max=12)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def knn_adj(c, k=5):
    xy = PANEL[c]["cent"]; X_ = np.c_[xy["lat"].values * 110.57, xy["lon"].values * 111.32 * np.cos(np.radians(xy["lat"].mean()))]
    D = ((X_[:, None] - X_[None]) ** 2).sum(-1); np.fill_diagonal(D, np.inf); A = len(X_); nb = np.zeros((A, A), np.float32)
    for i, js in enumerate(np.argsort(D, 1)[:, :k]): nb[i, js] = 1
    return nb / nb.sum(1, keepdims=True)

FC = {}
with part("II.2 forecasters × horizons × seeds"):
    SER = {c: series_tables(c) for c in CITIES_OK}
    AREA_GLOBAL = {f"{c}_{a}": i for i, (c, a) in enumerate((c, a) for c in CITIES_OK for a in PANEL[c]["areas"])}
    SEQ = {}
    for c in CITIES_OK:
        C = PANEL[c]["C"].astype(np.float32); NBC = np.einsum("ab,wbk->wak", knn_adj(c), C)
        SEQ[c] = {k: torch.as_tensor(v, device=DEVICE) for k, v in dict(x=np.log1p(C), at=np.log1p(C.sum(-1)), ct=np.log1p(C.sum(1)), nb=np.log1p(NBC)).items()}
    for H in HORIZONS:
        X, M = [], []
        for c in CITIES_OK:
            W = PANEL[c]["C"].shape[0]; Xc, Mc = build_rows(c, H, np.arange(52, W - H)); X.append(Xc); M.append(Mc)
        X = pd.concat(X, ignore_index=True); M = pd.concat(M, ignore_index=True)
        for col in ("city", "area"): M[col] = M[col].astype("category")
        tw = M["target_week"]
        tr = (tw <= TRAIN_END).values; ca = ((tw > TRAIN_END) & (tw <= VAL_END)).values; te = (tw > VAL_END).values
        es = tr & (tw > TRAIN_END - pd.Timedelta(weeks=26)).values; fit = tr & ~es
        log(f"H={H}: {len(M):,} rows ({fit.sum():,} fit, {es.sum():,} early stop, {ca.sum():,} validation, {te.sum():,} test)")
        for s in SEEDS_TREE:
            t = time.time(); m = fit_lgb_pois(X[fit], M["y"][fit], X[es], M["y"][es], seed=s); M[f"lgbm_s{s}"] = m.predict(X).astype(np.float32)
            log(f"  LightGBM seed {s}: {m.best_iteration_} trees, {time.time() - t:.0f}s")
            if H == 1 and s == SEEDS_TREE[0]:
                imp = pd.DataFrame({"feature": X.columns, "gain": m.booster_.feature_importance("gain")}).sort_values("gain", ascending=False)
                save_table(imp, "c03_lightgbm_feature_importance_H1")
            t = time.time(); m = fit_xgb_pois(X[fit], M["y"][fit], X[es], M["y"][es], seed=s); M[f"xgb_s{s}"] = m.predict(X).astype(np.float32)
            log(f"  XGBoost seed {s}: {m.best_iteration + 1} trees, {time.time() - t:.0f}s")
        if H in NN_HORIZONS:
            for kind in NEURAL:
                for s in SEEDS_NN:
                    if time_left() < 6.5 and s != SEEDS_NN[0]: log(f"  time budget: {kind} seed {s} skipped"); continue
                    t = time.time(); M[f"{kind}_s{s}"] = train_torch_rate(kind, X, M, fit, es, seed=s); log(f"  {kind} seed {s}: {time.time() - t:.0f}s")
        FC[H] = dict(M=M, X=X if H in (1, 4) else None, masks=dict(fit=fit, es=es, cal=ca, test=te))
        if H not in (1, 4): del X
        gc.collect()

    # ---- accuracy on validation and test, per seed
    def pred_cols(M, key): return [c_ for c_ in M.columns if re.fullmatch(fr"{key}_s\d+", c_)] if key in LEARNED else [key]
    ACC = []
    for H, d in FC.items():
        M = d["M"]
        for split, mk in (("validation", d["masks"]["cal"]), ("test", d["masks"]["test"])):
            for c in CITIES_OK:
                m = mk & (M["city"] == c).values; y_ = M["y"].values[m]
                for key in RATE_MODELS:
                    for col in pred_cols(M, key):
                        mu = M[col].values[m] + (0.05 if key in NAIVE else 0)
                        ACC.append(dict(H=H, split=split, city=c, model=RATE_MODELS[key], key=key, seed=int(col.split("_s")[1]) if "_s" in col else -1,
                                        poisson_deviance=pdev(y_, mu), mae=np.abs(y_ - M[col].values[m]).mean()))
    ACC = save_table(pd.DataFrame(ACC), "c04_forecast_accuracy_per_seed")
    # selection on the horizons where every learned model was run (mean over seeds and cities)
    vsel = (ACC[(ACC["split"] == "validation") & ACC["key"].isin(LEARNED) & ACC["H"].isin(NN_HORIZONS)]
            .groupby(["key", "H", "city"])["poisson_deviance"].mean().groupby("key").mean().sort_values())
    O3_SEL = vsel.index[0]
    vtree = ACC[(ACC["split"] == "validation") & ACC["key"].isin(TREES)].groupby("key")["poisson_deviance"].mean().sort_values()
    SEL_H = {H: (O3_SEL if any(c_.startswith(O3_SEL + "_s") for c_ in FC[H]["M"].columns) else vtree.index[0]) for H in HORIZONS}
    SEL_SEEDS = {H: sorted(int(c_.split("_s")[1]) for c_ in FC[H]["M"].columns if c_.startswith(SEL_H[H] + "_s")) for H in HORIZONS}
    save_table(vsel.rename("validation_deviance").reset_index(), "c04b_model_selection")
    log(f"validation deviance (mean over seeds, cities, horizons {NN_HORIZONS}): {vsel.round(4).to_dict()} → selected: {RATE_MODELS[O3_SEL]}; per horizon: {SEL_H}")
    ACCS = ACC.groupby(["H", "split", "city", "model", "key"]).agg(seeds=("seed", "size"), deviance=("poisson_deviance", "mean"), deviance_sd=("poisson_deviance", "std"),
                                                                    mae=("mae", "mean"), mae_sd=("mae", "std")).reset_index()
    save_table(ACCS, "c05_forecast_accuracy_mean_sd")
    display(ACCS[ACCS["split"] == "test"].pivot_table(index=["H", "model"], columns="city", values="deviance").round(4))

    # ---- skill of the selected model vs every baseline, separately per seed
    SKILL = []
    for H, d in FC.items():
        M = d["M"]
        for c in CITIES_OK:
            m = d["masks"]["test"] & (M["city"] == c).values; y_ = M["y"].values[m]; wk = M["target_week"].values[m]
            for s in SEL_SEEDS[H]:
                sc = f"{SEL_H[H]}_s{s}"
                dsel = unit_dev(y_, M[sc].values[m])
                for base in RATE_MODELS:
                    if base == SEL_H[H]: continue
                    bcol = f"{base}_s{s}" if base in LEARNED else base
                    if bcol not in M: continue
                    est, lo, hi = block_boot(unit_dev(y_, M[bcol].values[m] + (0.05 if base in NAIVE else 0)) - dsel, wk)
                    SKILL.append(dict(H=H, city=c, seed=s, selected=RATE_MODELS[SEL_H[H]], baseline=RATE_MODELS[base], baseline_family="naive/statistical" if base in NAIVE else "learned",
                                      deviance_gain=est, ci_lo=lo, ci_hi=hi, significant=lo > 0))
    SKILL = save_table(pd.DataFrame(SKILL), "c06_skill_vs_every_baseline_per_seed")
    SKS = SKILL.groupby(["H", "city", "baseline", "baseline_family"]).agg(seeds=("seed", "size"), gain=("deviance_gain", "mean"), min_ci_lo=("ci_lo", "min"),
                                                                       significant_all_seeds=("significant", "all")).reset_index()
    save_table(SKS, "c07_skill_summary"); display(SKS.round(4))
    for H in (1, 4):                                                   # test predictions for later analysis
        M = FC[H]["M"]; te = FC[H]["masks"]["test"]
        keep = ["city", "area", "type", "target_week", "y", "ma4", "naive"] + [c_ for c_ in M.columns if c_.startswith(SEL_H[H] + "_s")]
        M.loc[te, keep].to_csv(PRED / f"weekly_test_predictions_H{H}.csv.gz", index=False, float_format="%.4f")
    fig, ax = plt.subplots(1, len(CITIES_OK), figsize=(5.2 * len(CITIES_OK), 4.2), sharey=False)
    ax = np.atleast_1d(ax)
    for i, c in enumerate(CITIES_OK):
        g = ACCS[(ACCS["split"] == "test") & (ACCS["city"] == c)]
        for key in ["lgbm", "xgb", "stgru", "gru", "glm", "ses", "ma4", "snaive"]:
            gg = g[g["key"] == key].sort_values("H")
            if len(gg): ax[i].errorbar(gg["H"], gg["deviance"], yerr=gg["deviance_sd"].fillna(0), marker="o", capsize=3, label=RATE_MODELS[key])
        ax[i].set_title(f"{CITY_NAME[c]}: test deviance by horizon"); ax[i].set_xlabel("horizon (weeks)"); ax[i].set_xticks(HORIZONS)
    ax[0].set_ylabel("Poisson deviance (lower is better)"); ax[-1].legend(fontsize=7); savefig("figC2_deviance_by_horizon")

## Part III. 90 % prediction intervals
All intervals are built around the selected forecaster and calibrated on the January–June 2024 block only; coverage is measured on the test period, overall, per fifth of series volume (mean weekly count in training) and per area.

| Method | Idea |
|---|---|
| Poisson, negative binomial | parametric quantiles of the forecast mean (NB size by moments on the calibration block) |
| Split conformal (CP) | one quantile of the normalised score \|y − μ\| / √(μ + 0.5) |
| **Mondrian conformal** (k = 3, 5, 10) | the same score, one quantile per bin of the forecast size (bins = quantiles of μ on the calibration block) |
| CQR | conformalised quantile regression: LightGBM 5 % and 95 % quantile models, widened by a calibrated margin |
| Locally weighted CP | score divided by a learned spread σ̂(x) (LightGBM on \|y − μ\| of the early-stopping block) |
| ACI, Mondrian-ACI | adaptive conformal inference (Gibbs & Candès 2021): the miscoverage level is updated every week from the coverage errors that are already known (delay = horizon), γ = 0.05 |

In [ ]:
ACI_GAMMA = 0.05
def aci_intervals(mu, y, weeks, cal_scores, H, bins=None, cal_bins=None, gamma=ACI_GAMMA):
    """Adaptive conformal inference with a delay of H weeks. Optional Mondrian bins (one alpha per bin)."""
    mu = np.asarray(mu, float); y = np.asarray(y, float); s = np.sqrt(mu + 0.5)
    bins = np.zeros(len(mu), int) if bins is None else bins; cal_bins = np.zeros(len(cal_scores), int) if cal_bins is None else cal_bins
    ub = np.unique(np.r_[bins, cal_bins]); alpha = {b: ALPHA for b in ub}; sorted_cal = {b: np.sort(cal_scores[cal_bins == b]) for b in ub}
    lo = np.zeros(len(mu)); hi = np.zeros(len(mu)); uw = np.sort(np.unique(weeks)); errs = {b: {} for b in ub}
    for i, w in enumerate(uw):
        m = weeks == w
        for b in ub:
            mb = m & (bins == b)
            if not mb.any(): continue
            known = [errs[b][uw[j]] for j in range(max(0, i - 400), i - H + 1) if uw[j] in errs[b]]   # errors of weeks already observed
            a = ALPHA
            for e in known: a = a + gamma * (ALPHA - e)
            alpha[b] = a; sc = sorted_cal[b] if len(sorted_cal[b]) else np.sort(cal_scores)
            if a <= 0: q = np.inf
            elif a >= 1: q = 0.0
            else:
                n = len(sc); lev = min(1.0, np.ceil((n + 1) * (1 - a)) / n); q = float(np.quantile(sc, lev, method="higher"))
            l_, h_ = int_from_q(mu[mb], q if np.isfinite(q) else 1e6)
            lo[mb], hi[mb] = l_, h_
            errs[b][w] = float(1 - ((y[mb] >= l_) & (y[mb] <= h_)).mean())
    return lo, hi

def interval_suite(H, c, mu_cal, y_cal, mu_t, y_t, weeks_t, extra=None):
    """All interval methods for one horizon and city. `extra` holds CQR quantiles and the locally weighted spread if available."""
    out = {}
    out["Poisson"] = parametric_interval(mu_t, "Poisson"); out["Neg. binomial"] = parametric_interval(mu_t, "NB", nb_size(mu_cal, y_cal))
    sc = norm_score(y_cal, mu_cal); out["Split conformal (CP)"] = int_from_q(mu_t, conformal_q(sc))
    for k in (3, 5, 10):
        e = mondrian_edges(mu_cal, k); bc = np.digitize(mu_cal, e); qb = np.array([conformal_q(sc[bc == i]) for i in range(len(e) + 1)])
        out[f"Mondrian CP (k={k})" if k != 5 else "Mondrian CP"] = int_from_q(mu_t, qb[np.digitize(mu_t, e)])
    e5 = mondrian_edges(mu_cal, 5)
    out["ACI"] = aci_intervals(mu_t, y_t, weeks_t, sc, H)
    out["Mondrian-ACI"] = aci_intervals(mu_t, y_t, weeks_t, sc, H, bins=np.digitize(mu_t, e5), cal_bins=np.digitize(mu_cal, e5))
    if extra is not None:
        if "q_lo_cal" in extra:
            E = np.maximum(extra["q_lo_cal"] - y_cal, y_cal - extra["q_hi_cal"]); Q = conformal_q(E)
            lo = np.maximum(np.ceil(extra["q_lo_t"] - Q), 0); hi = np.floor(extra["q_hi_t"] + Q); bad = hi < lo
            out["CQR"] = (np.where(bad, np.round(mu_t), lo), np.where(bad, np.round(mu_t), hi))
        if "sig_cal" in extra:
            s_cal = np.maximum(extra["sig_cal"], 0.1); s_t = np.maximum(extra["sig_t"], 0.1)
            q = conformal_q(np.abs(y_cal - mu_cal) / s_cal); out["Locally weighted CP"] = int_from_q(mu_t, q, scale=s_t)
    return out

def series_volume_quintiles(c, train_end):
    C = PANEL[c]["C"]; v = C[PANEL[c]["weeks"] <= train_end].mean(0)
    V = pd.DataFrame({"area": np.repeat(PANEL[c]["areas"], len(FINE)), "type": np.tile(np.arange(len(FINE)), len(PANEL[c]["areas"])).astype(np.int8), "vol": v.reshape(-1)})
    V["vq"] = pd.qcut(V["vol"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int); return V

COV, IV = [], {}
with part("III. intervals on the main split"):
    for H in HORIZONS:
        d = FC[H]; M = d["M"]; X = d["X"]; s0 = SEL_SEEDS[H][0]; scol = f"{SEL_H[H]}_s{s0}"
        extra_all = None
        if X is not None and time_left() > 5.0:                       # CQR and locally weighted CP need the feature matrix (H = 1 and 4)
            fit, es, cal, te = (d["masks"][k] for k in ("fit", "es", "cal", "test")); sel = cal | te; t = time.time()
            ql = fit_lgb_quantile(X[fit], M["y"][fit], X[es], M["y"][es], ALPHA / 2); qh = fit_lgb_quantile(X[fit], M["y"][fit], X[es], M["y"][es], 1 - ALPHA / 2)
            Xr = X.assign(mu=M[scol].values); rs = fit_lgb_l2(Xr[es], np.abs(M["y"][es] - M[scol][es]), Xr[es], np.abs(M["y"][es] - M[scol][es]))
            extra_all = dict(q_lo=np.full(len(M), np.nan), q_hi=np.full(len(M), np.nan), sig=np.full(len(M), np.nan))
            extra_all["q_lo"][sel] = ql.predict(X[sel]); extra_all["q_hi"][sel] = qh.predict(X[sel]); extra_all["sig"][sel] = rs.predict(Xr[sel])
            log(f"  H={H}: CQR and locally weighted CP models in {time.time() - t:.0f}s")
        for c in CITIES_OK:
            mc = d["masks"]["cal"] & (M["city"] == c).values; mt = d["masks"]["test"] & (M["city"] == c).values
            if mc.sum() == 0 or mt.sum() == 0: continue
            T = M.loc[mt, ["area", "type", "target_week", "y", scol]].reset_index(drop=True).merge(series_volume_quintiles(c, TRAIN_END), on=["area", "type"], how="left")
            ex = None
            if extra_all is not None:
                ex = dict(q_lo_cal=extra_all["q_lo"][mc], q_hi_cal=extra_all["q_hi"][mc], q_lo_t=extra_all["q_lo"][mt], q_hi_t=extra_all["q_hi"][mt],
                          sig_cal=extra_all["sig"][mc], sig_t=extra_all["sig"][mt])
            res = interval_suite(H, c, M[scol].values[mc], M["y"].values[mc], T[scol].values, T["y"].values, T["target_week"].values, ex)
            for meth, (lo, hi) in res.items():
                COV.append(dict(H=H, city=c, method=meth, forecaster=RATE_MODELS[SEL_H[H]], seed=s0, **coverage_report(T["y"].values, lo, hi, T["vq"].values, T["area"].values)))
            if H == 1: IV[c] = T.assign(**{f"lo|{m}": v[0] for m, v in res.items()}, **{f"hi|{m}": v[1] for m, v in res.items()})
        # robustness of CP vs Mondrian over the seeds of the forecaster
        for s in SEL_SEEDS[H][1:]:
            col = f"{SEL_H[H]}_s{s}"
            for c in CITIES_OK:
                mc = d["masks"]["cal"] & (M["city"] == c).values; mt = d["masks"]["test"] & (M["city"] == c).values
                T = M.loc[mt, ["area", "type", "y", col]].reset_index(drop=True).merge(series_volume_quintiles(c, TRAIN_END), on=["area", "type"], how="left")
                sc = norm_score(M["y"].values[mc], M[col].values[mc]); e = mondrian_edges(M[col].values[mc]); bc = np.digitize(M[col].values[mc], e)
                qb = np.array([conformal_q(sc[bc == i]) for i in range(len(e) + 1)])
                for meth, (lo, hi) in {"Split conformal (CP)": int_from_q(T[col].values, conformal_q(sc)), "Mondrian CP": int_from_q(T[col].values, qb[np.digitize(T[col].values, e)])}.items():
                    COV.append(dict(H=H, city=c, method=meth, forecaster=RATE_MODELS[SEL_H[H]], seed=s, **coverage_report(T["y"].values, lo, hi, T["vq"].values, T["area"].values)))
        log(f"  intervals H={H} done")
    COV = save_table(pd.DataFrame(COV), "d01_interval_coverage_main_split")
    show = COV[COV["seed"] == COV.groupby(["H", "city"])["seed"].transform("min")]
    display(show[["H", "city", "method", "coverage", "width", "interval_score", "cov_Q1", "cov_Q3", "cov_Q5", "max_abs_dev_from_90", "areas_within_5pts"]].round(3))
    save_table(COV.groupby(["H", "city", "method"])[["coverage", "max_abs_dev_from_90", "width"]].agg(["mean", "std", "count"]).reset_index(), "d02_interval_coverage_over_seeds")
    for c in CITIES_OK:
        if c in IV: IV[c].to_csv(PRED / f"intervals_test_H1_{c}.csv.gz", index=False, float_format="%.3f")
    main_methods = ["Poisson", "Neg. binomial", "Split conformal (CP)", "Mondrian CP", "CQR", "Locally weighted CP", "Mondrian-ACI"]
    fig, ax = plt.subplots(1, len(CITIES_OK), figsize=(5.3 * len(CITIES_OK), 4.3), sharey=True); ax = np.atleast_1d(ax)
    for i, c in enumerate(CITIES_OK):
        g = show[(show["H"] == 1) & (show["city"] == c)]
        for meth, st in zip(main_methods, [":", "-.", "--", "-", (0, (3, 1, 1, 1)), (0, (1, 1)), (0, (5, 2))]):
            r = g[g["method"] == meth]
            if len(r): ax[i].plot(range(1, 6), [100 * r.iloc[0][f"cov_Q{q}"] for q in range(1, 6)], linestyle=st, marker="o", label=meth, lw=2 if meth == "Mondrian CP" else 1)
        ax[i].axhline(90, color="k", lw=.8, ls=":"); ax[i].set_xticks(range(1, 6)); ax[i].set_xticklabels(["Q1\nquiet", "Q2", "Q3", "Q4", "Q5\nbusy"])
        ax[i].set_title(f"{CITY_NAME[c]}: coverage by series volume (H = 1)")
    ax[0].set_ylabel("coverage of 90 % interval (%)"); ax[-1].legend(fontsize=7); savefig("figD1_coverage_by_volume_quintile")
    fig, ax = plt.subplots(figsize=(11, 4.5)); piv = show.pivot_table(index="method", columns=["city", "H"], values="max_abs_dev_from_90") * 100
    piv = piv.loc[piv.mean(1).sort_values().index]; im = ax.imshow(piv.values, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=12)
    ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index, fontsize=8); ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels([f"{a} H{b}" for a, b in piv.columns], fontsize=7, rotation=45)
    for (i_, j_), v in np.ndenumerate(piv.values): ax.text(j_, i_, f"{v:.1f}", ha="center", va="center", fontsize=7)
    plt.colorbar(im, ax=ax, label="worst volume-quintile deviation from 90 % (points)"); ax.set_title("Worst-quintile coverage error (3 points = A6 threshold)")
    savefig("figD2_worst_quintile_deviation_heatmap")

## Part IV. Rolling origins: every test year from 2020
For each test year *Y* the global LightGBM-Poisson forecaster (seed 0, *H* = 1) is re-trained on all target weeks before the 26-week calibration block that precedes *Y* (its last 13 weeks for early stopping), calibrated on that block and tested on the 52 weeks of *Y*. 2020 contains the COVID-19 shift. A city enters a fold only if it has at least 26 training weeks.

In [ ]:
ROLL_COV, ROLL_SKILL = [], []
with part("IV. rolling-origin folds"):
    M1 = FC[1]["M"]; X1 = FC[1]["X"]; tw = M1["target_week"]
    for Y in ROLL_YEARS:
        if time_left() < 4.2: log(f"  time budget: fold {Y} skipped"); continue
        t0 = pd.Timestamp(f"{Y}-01-01"); cal_start = t0 - pd.Timedelta(weeks=26)
        te = ((tw >= t0) & (tw < pd.Timestamp(f"{Y + 1}-01-01"))).values; ca = ((tw >= cal_start) & (tw < t0)).values; tr = (tw < cal_start).values
        es = tr & (tw >= cal_start - pd.Timedelta(weeks=13)).values; fit = tr & ~es
        if te.sum() == 0 or fit.sum() == 0: continue
        t = time.time(); m = fit_lgb_pois(X1[fit], M1["y"][fit], X1[es], M1["y"][es], seed=0); sel = ca | te | es
        mu = np.full(len(M1), np.nan, np.float32); mu[sel] = m.predict(X1[sel])
        ex_all = None
        if time_left() > 4.8:
            ql = fit_lgb_quantile(X1[fit], M1["y"][fit], X1[es], M1["y"][es], ALPHA / 2); qh = fit_lgb_quantile(X1[fit], M1["y"][fit], X1[es], M1["y"][es], 1 - ALPHA / 2)
            Xr = X1.assign(mu=mu); rs = fit_lgb_l2(Xr[es], np.abs(M1["y"][es] - mu[es]), Xr[es], np.abs(M1["y"][es] - mu[es]))
            ex_all = dict(q_lo=np.full(len(M1), np.nan), q_hi=np.full(len(M1), np.nan), sig=np.full(len(M1), np.nan))
            s2 = ca | te; ex_all["q_lo"][s2] = ql.predict(X1[s2]); ex_all["q_hi"][s2] = qh.predict(X1[s2]); ex_all["sig"][s2] = rs.predict(Xr[s2])
        log(f"  fold {Y}: models in {time.time() - t:.0f}s")
        for c in CITIES_OK:
            cm = (M1["city"] == c).values
            if (tr & cm).sum() == 0 or len(np.unique(tw[tr & cm])) < 26: log(f"    {c}: fewer than 26 training weeks before {Y}, skipped"); continue
            mc, mt = ca & cm, te & cm
            if mc.sum() == 0 or mt.sum() == 0: continue
            y_t = M1["y"].values[mt]; wk = tw.values[mt]
            for base in ("ma4", "ses", "snaive"):
                est, lo_, hi_ = block_boot(unit_dev(y_t, M1[base].values[mt] + 0.05) - unit_dev(y_t, mu[mt]), wk)
                ROLL_SKILL.append(dict(test_year=Y, city=c, baseline=RATE_MODELS[base], deviance_gain=est, ci_lo=lo_, ci_hi=hi_, significant=lo_ > 0,
                                       deviance_model=pdev(y_t, mu[mt]), deviance_baseline=pdev(y_t, M1[base].values[mt] + 0.05)))
            T = M1.loc[mt, ["area", "type"]].reset_index(drop=True).merge(series_volume_quintiles(c, cal_start), on=["area", "type"], how="left")
            ex = None if ex_all is None else dict(q_lo_cal=ex_all["q_lo"][mc], q_hi_cal=ex_all["q_hi"][mc], q_lo_t=ex_all["q_lo"][mt], q_hi_t=ex_all["q_hi"][mt],
                                                   sig_cal=ex_all["sig"][mc], sig_t=ex_all["sig"][mt])
            res = interval_suite(1, c, mu[mc], M1["y"].values[mc], mu[mt], y_t, wk, ex)
            for meth, (lo, hi) in res.items():
                ROLL_COV.append(dict(test_year=Y, city=c, method=meth, **coverage_report(y_t, lo, hi, T["vq"].values, T["area"].values)))
        save_table(pd.DataFrame(ROLL_COV), "e01_rolling_interval_coverage"); save_table(pd.DataFrame(ROLL_SKILL), "e02_rolling_forecast_skill")
    ROLL_COV = pd.DataFrame(ROLL_COV); ROLL_SKILL = pd.DataFrame(ROLL_SKILL)
    if len(ROLL_COV):
        display(ROLL_COV.pivot_table(index="method", columns=["city", "test_year"], values="max_abs_dev_from_90").mul(100).round(1))
        display(ROLL_COV.pivot_table(index="method", columns=["city", "test_year"], values="coverage").mul(100).round(1))
        display(ROLL_SKILL.round(4))
        fig, ax = plt.subplots(2, len(CITIES_OK), figsize=(5.3 * len(CITIES_OK), 7.5), sharex=True); ax = np.atleast_2d(ax)
        if ax.shape[0] == 1: ax = ax.T
        for i, c in enumerate(CITIES_OK):
            g = ROLL_COV[ROLL_COV["city"] == c]
            for meth in ["Split conformal (CP)", "Mondrian CP", "CQR", "Locally weighted CP", "ACI", "Mondrian-ACI", "Neg. binomial"]:
                gg = g[g["method"] == meth].sort_values("test_year")
                if len(gg):
                    ax[0, i].plot(gg["test_year"], 100 * gg["max_abs_dev_from_90"], marker="o", label=meth, lw=2 if meth == "Mondrian CP" else 1)
                    ax[1, i].plot(gg["test_year"], 100 * gg["coverage"], marker="o", label=meth, lw=2 if meth == "Mondrian CP" else 1)
            ax[0, i].axhline(3, color="k", ls=":", lw=.8); ax[1, i].axhline(90, color="k", ls=":", lw=.8)
            ax[0, i].set_title(f"{CITY_NAME[c]}: worst-quintile deviation (points)"); ax[1, i].set_title(f"{CITY_NAME[c]}: overall coverage (%)")
        ax[0, -1].legend(fontsize=7); savefig("figE1_rolling_origin_calibration")

## Part V. Equity check with census data
Area demographics come from the **American Community Survey 5-year estimates (2022)**, tract level, from the Census API (no key needed for this volume): population, poverty (B17001), race/ethnicity (B03002) and median household income (B19013). Tract centroids come from the Census Gazetteer files. Each tract is assigned to the area (community area, precinct, neighbourhood) of the majority of its 25 nearest geolocated incidents of the last two years; tracts whose nearest incident is more than 1 km away (water, parks, airports) are dropped. Area values are population-weighted.

Areas are split per city into thirds by **poverty rate**, and into **majority-minority** (non-Hispanic white share below 50 %) vs other. For each group we report the coverage of the 90 % intervals (H = 1, test period), the forecast skill over the 4-week average, and how the top-10 risk lists distribute attention.

In [ ]:
ACS_VARS = {"B01003_001E": "pop", "B17001_001E": "pov_universe", "B17001_002E": "pov", "B03002_001E": "race_total", "B03002_003E": "nh_white", "B19013_001E": "median_income"}
COUNTIES = {"CHI": [("17", "031")], "NYC": [("36", "005"), ("36", "047"), ("36", "061"), ("36", "081"), ("36", "085")], "SF": [("06", "075")]}
def http_get(url, timeout=120):
    import urllib.request
    for a in range(4):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r: return r.read()
        except Exception as e: last = e; time.sleep(5 * (a + 1))
    raise RuntimeError(f"{url}: {last!r}")
def acs_tracts(state, county):
    f = DATA / f"acs2022_{state}{county}.csv"
    if not f.exists():
        url = f"https://api.census.gov/data/2022/acs/acs5?get={','.join(ACS_VARS)}&for=tract:*&in=state:{state}&in=county:{county}"
        js = json.loads(http_get(url)); pd.DataFrame(js[1:], columns=js[0]).to_csv(f, index=False)
    d = pd.read_csv(f, dtype=str); d["GEOID"] = d["state"].str.zfill(2) + d["county"].str.zfill(3) + d["tract"].str.zfill(6)
    for v, n in ACS_VARS.items(): d[n] = pd.to_numeric(d[v], errors="coerce")
    d.loc[d["median_income"] < 0, "median_income"] = np.nan
    return d[["GEOID"] + list(ACS_VARS.values())]
def gazetteer(state):
    f = DATA / f"gaz_tracts_{state}.txt"
    if not f.exists():
        last = None
        for url in (f"https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2022_Gazetteer/2022_gaz_tracts_{state}.txt",
                    f"https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2023_Gazetteer/2023_gaz_tracts_{state}.txt",
                    f"https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2020_Gazetteer/2020_gaz_tracts_{state}.txt"):
            try: f.write_bytes(http_get(url)); break
            except Exception as e: last = e
        else: raise RuntimeError(f"gazetteer for state {state} not available: {last}")
    g = pd.read_csv(f, sep="\t", dtype=str); g.columns = [c_.strip() for c_ in g.columns]
    return pd.DataFrame({"GEOID": g["GEOID"].str.zfill(11), "lat": pd.to_numeric(g["INTPTLAT"]), "lon": pd.to_numeric(g["INTPTLONG"])})
def tracts_to_areas(c, tr):
    from sklearn.neighbors import BallTree
    P = GEO[c]; tree = BallTree(np.radians(P[["lat", "lon"]].values.astype(float)), metric="haversine")
    dist, idx = tree.query(np.radians(tr[["lat", "lon"]].values), k=25); lab = np.asarray(P["area"].astype(str), dtype=object)[idx]
    tr = tr.copy(); tr["area"] = [pd.Series(r).mode().iloc[0] for r in lab]; tr["nearest_km"] = dist[:, 0] * 6371.0
    return tr[tr["nearest_km"] <= 1.0]

DEMO, EQ_COV, EQ_SKILL, EQ_RISK, AREA_EQ = [], [], [], [], []
with part("V. equity check (census)"):
    for c in CITIES_OK:
        try:
            tr = pd.concat([acs_tracts(s, k) for s, k in COUNTIES[c]]).merge(pd.concat([gazetteer(s) for s in sorted({s for s, _ in COUNTIES[c]})]), on="GEOID")
            tr = tracts_to_areas(c, tr[tr["pop"] > 0])
        except Exception as e:
            log(f"  census data for {c} not available: {str(e)[:200]}"); continue
        g = tr.groupby("area").apply(lambda d: pd.Series(dict(tracts=len(d), pop=d["pop"].sum(), poverty_rate=d["pov"].sum() / max(d["pov_universe"].sum(), 1),
                                                             minority_share=1 - d["nh_white"].sum() / max(d["race_total"].sum(), 1),
                                                             median_income=np.average(d["median_income"].fillna(d["median_income"].median()), weights=d["pop"]))))
        g = g.reset_index().assign(city=c); g["poverty_group"] = pd.qcut(g["poverty_rate"].rank(method="first"), 3, labels=["low poverty", "middle", "high poverty"]).astype(str)
        g["majority_minority"] = np.where(g["minority_share"] > 0.5, "majority-minority", "white-majority or mixed"); DEMO.append(g)
        log(f"  {c}: {len(tr)} tracts → {len(g)} of {len(PANEL[c]['areas'])} areas with census data")
        if c not in IV: continue
        T = IV[c].merge(g[["area", "poverty_group", "majority_minority", "poverty_rate", "minority_share"]], on="area", how="inner")
        meths = [m[3:] for m in T.columns if m.startswith("lo|")]
        for gcol in ("poverty_group", "majority_minority"):
            for gv, gg in T.groupby(gcol):
                for meth in meths:
                    cov = (gg["y"] >= gg[f"lo|{meth}"]) & (gg["y"] <= gg[f"hi|{meth}"])
                    EQ_COV.append(dict(city=c, grouping=gcol, group=gv, method=meth, rows=len(gg), areas=gg["area"].nunique(), coverage=cov.mean(),
                                       mean_width=(gg[f"hi|{meth}"] - gg[f"lo|{meth}"]).mean(), mean_count=gg["y"].mean()))
        for meth in meths:                                             # per-area coverage vs demographics
            cov = ((T["y"] >= T[f"lo|{meth}"]) & (T["y"] <= T[f"hi|{meth}"])).groupby(T["area"]).mean().rename("coverage").reset_index().merge(g, on="area")
            rho_p = spearmanr(cov["coverage"], cov["poverty_rate"]); rho_m = spearmanr(cov["coverage"], cov["minority_share"])
            AREA_EQ.append(dict(city=c, method=meth, areas=len(cov), spearman_cov_poverty=rho_p[0], p_poverty=rho_p[1], spearman_cov_minority=rho_m[0], p_minority=rho_m[1],
                                sd_area_coverage=cov["coverage"].std()))
        # forecast skill over the 4-week average, and risk lists, by group (H = 1, seed 0)
        M = FC[1]["M"]; mt = FC[1]["masks"]["test"] & (M["city"] == c).values; scol = f"{SEL_H[1]}_s{SEL_SEEDS[1][0]}"
        Tt = M.loc[mt, ["area", "type", "target_week", "y", "ma4", scol]].merge(g[["area", "poverty_group", "majority_minority"]], on="area", how="inner")
        for gcol in ("poverty_group", "majority_minority"):
            for gv, gg in Tt.groupby(gcol):
                est, lo_, hi_ = block_boot(unit_dev(gg["y"].values, gg["ma4"].values + 0.05) - unit_dev(gg["y"].values, gg[scol].values), gg["target_week"].values)
                EQ_SKILL.append(dict(city=c, grouping=gcol, group=gv, deviance_model=pdev(gg["y"], gg[scol]), deviance_ma4=pdev(gg["y"], gg["ma4"] + 0.05),
                                     gain=est, ci_lo=lo_, ci_hi=hi_, mean_count=gg["y"].mean(), rel_mae=np.abs(gg["y"] - gg[scol]).mean() / max(gg["y"].mean(), 1e-6)))
        k10 = min(10, len(PANEL[c]["areas"]) // 4)
        R_ = []
        for (w, k), gg in Tt.groupby(["target_week", "type"]):
            if (gg["y"] > 0).sum() < k10: continue
            top_true = set(gg.nlargest(k10, "y")["area"]); listed = gg.nlargest(k10, scol)
            for gv, l_ in listed.groupby("poverty_group"):
                R_.append(dict(group=gv, listed=len(l_), hits=int(l_["area"].isin(top_true).sum()), true_top=int(gg[gg["area"].isin(top_true)]["poverty_group"].eq(gv).sum())))
        if R_:
            R_ = pd.DataFrame(R_).groupby("group")[["listed", "hits", "true_top"]].sum().reset_index()
            R_["share_of_list_slots"] = R_["listed"] / R_["listed"].sum(); R_["share_of_true_top"] = R_["true_top"] / R_["true_top"].sum()
            R_["precision"] = R_["hits"] / R_["listed"]; EQ_RISK.append(R_.assign(city=c, list_size=k10))
    if DEMO: save_table(pd.concat(DEMO), "f01_area_demographics")
    EQ_COV = save_table(pd.DataFrame(EQ_COV), "f02_coverage_by_group"); EQ_SKILL = save_table(pd.DataFrame(EQ_SKILL), "f03_forecast_skill_by_group")
    if EQ_RISK: save_table(pd.concat(EQ_RISK), "f04_risk_list_attention_by_poverty_group")
    AREA_EQ = save_table(pd.DataFrame(AREA_EQ), "f05_area_coverage_vs_demographics")
    if len(EQ_COV):
        GAP = []
        for (c, gcol, meth), gg in EQ_COV.groupby(["city", "grouping", "method"]):
            v = gg.set_index("group")["coverage"]
            hi_lab, lo_lab = ("high poverty", "low poverty") if gcol == "poverty_group" else ("majority-minority", "white-majority or mixed")
            if hi_lab in v and lo_lab in v: GAP.append(dict(city=c, grouping=gcol, method=meth, coverage_disadvantaged=v[hi_lab], coverage_advantaged=v[lo_lab], gap=v[hi_lab] - v[lo_lab]))
        GAP = save_table(pd.DataFrame(GAP), "f06_coverage_gap_between_groups"); display(GAP.round(3))
        if len(GAP):
            fig, ax = plt.subplots(figsize=(10, 4.5)); gp = GAP[GAP["grouping"] == "poverty_group"].pivot_table(index="method", columns="city", values="gap") * 100
            gp.plot.barh(ax=ax); ax.axvline(0, color="k", lw=.8); ax.set_xlabel("coverage(high-poverty third) − coverage(low-poverty third), points")
            ax.set_title("Coverage gap between the poorest and the richest third of areas (H = 1, test)"); savefig("figF1_coverage_gap_poverty")
    display(EQ_SKILL.round(4)); display(AREA_EQ.round(3))

### I.2 (run last) Forecasts and re-runs on the benchmark
This cell runs after the city experiments because the re-runs of the deep models have the least predictable run time; each one is skipped if the remaining time budget is too small, and the first seed of every model in both cities runs before any extra seed.

In [ ]:
with part("I.2 benchmark: forecasts, re-runs and protocols"):
    prep_sthsl()
    try: STSHN_DIR = prep_stshn(); STSHN_OK = True
    except Exception as e: log(f"ST-SHN not available: {e}"); STSHN_OK = False
    BPRED, BRUNS = {}, []                                             # BPRED[city][model] = list of test prediction vectors (one per seed)
    BENCH_DATA = {}
    for c in BENCH_CITIES:
        Xall, ntr, nva, nte = bench_load(c); R, Cc, T, K = Xall.shape; N = R * Cc
        trd, vad, ted = np.arange(30, ntr), np.arange(ntr, ntr + nva), np.arange(ntr + nva, T)
        (Xtr, ytr), (Xva, yva), (Xte, yte) = (bench_features(Xall, d) for d in (trd, vad, ted))
        cat = Xte["cat"].cat.codes.values; act = (Xall[:, :, :ntr] > 0).mean(2).reshape(N, K); tl = ((act > 0.2) & (act < 0.8)).reshape(-1)
        keep2 = np.tile(tl, nte) & (yte > 0)
        BENCH_DATA[c] = dict(yte=yte, yva=yva, cat=cat, K=K, keep2=keep2, nte=nte, N=N, Xall=Xall, ntr=ntr, Xte=Xte, Xva=Xva)
        P = BPRED.setdefault(c, {}); PV = BENCH_DATA[c].setdefault("val_preds", {}); t = time.time()
        for s in SEEDS_BENCH:                                         # ours, several seeds
            ml = fit_lgb_pois(Xtr, ytr, Xva, yva, seed=s); mx = fit_xgb_pois(Xtr, ytr, Xva, yva, seed=s)
            P.setdefault("Ours: GBM-Poisson (LightGBM)", []).append(ml.predict(Xte)); PV.setdefault("Ours: GBM-Poisson (LightGBM)", []).append(ml.predict(Xva))
            P.setdefault("Ours: GBM-Poisson (XGBoost)", []).append(mx.predict(Xte)); PV.setdefault("Ours: GBM-Poisson (XGBoost)", []).append(mx.predict(Xva))
        vdev = {n: np.mean([pdev(yva, v) for v in PV[n]]) for n in ("Ours: GBM-Poisson (LightGBM)", "Ours: GBM-Poisson (XGBoost)")}
        BENCH_DATA[c]["ours"] = OURS = min(vdev, key=vdev.get)
        log(f"{c}: our forecasters ({len(SEEDS_BENCH)} seeds) in {time.time() - t:.0f}s; validation deviance {vdev} → {OURS}")
        P["Ours – protocol-aware (zero-truncated median)"] = [ztp_median(p) for p in P[OURS]]
        P["Naive: last day"] = [Xte["lag1"].values]; P["Naive: 30-day mean"] = [Xte["m30"].values]; P["Trivial: constant 1"] = [np.ones_like(yte)]

    def bench_job(kind, c, s):
        yte = BENCH_DATA[c]["yte"]; P = BPRED[c]
        if os.environ.get("OA_SKIP_ST") == "1" and kind in ("ST-HSL", "ST-SHN"): return      # local testing only
        if kind == "ST-HSL":
            r = run_sthsl(c, s, 2 if FAST else 25, timeout_s=int(min(15 if FAST else 100, 60 * (time_left() - 0.5)) * 60))
            if r is None: return
            pt, pv, info = r; pt = pt.reshape(-1)
            if len(pt) != len(yte): log("  ST-HSL output does not line up with the test set; skipped"); return
            P.setdefault("ST-HSL (re-run)", []).append(pt); BRUNS.append(info)
        elif kind == "ST-SHN":
            if not STSHN_OK: return
            r = run_stshn(c, s, timeout_s=int(min(20 if FAST else 120, 60 * (time_left() - 0.5)) * 60), epochs=1 if FAST else None)
            if r is None: return
            pv_, pt_, info = r
            P.setdefault("ST-SHN (re-run, epoch chosen on validation)", []).append(pv_.reshape(-1))
            P.setdefault("ST-SHN (re-run, as its code reports: best test epoch)", []).append(pt_.reshape(-1)); BRUNS.append(info)
        else:
            pt, pv, info = train_graph_model(kind, c, s); P.setdefault(f"{kind} (official model code, re-trained)", []).append(pt.reshape(-1)); BRUNS.append(info)
    # first one run of every model in both cities, then the extra seeds (so a time-out never leaves a model without any run)
    JOBS = [(k, c, SEEDS_STSHN[0] if k == "ST-SHN" else SEEDS_BENCH[0]) for k in ("Graph WaveNet", "MTGNN", "ST-HSL", "ST-SHN") for c in BENCH_CITIES]
    JOBS += [(k, c, s) for k in ("Graph WaveNet", "MTGNN", "ST-HSL") for c in BENCH_CITIES for s in SEEDS_BENCH[1:]]
    JOBS += [("ST-SHN", c, s) for c in BENCH_CITIES for s in SEEDS_STSHN[1:]]
    for kind, c, s in JOBS:
        if time_left() < 0.7: log(f"  time budget: {kind} {c} seed {s} skipped"); continue
        try: bench_job(kind, c, s)
        except Exception as e: log(f"  {kind} {c} seed {s} failed: {str(e)[:300]}"); open(LOGS / "errors.log", "a").write(f"{kind} {c} {s}\n{traceback.format_exc()}\n")
        for c_ in BENCH_CITIES:
            np.savez_compressed(PRED / f"benchmark_{c_}_test_predictions.npz", y=BENCH_DATA[c_]["yte"], cat=BENCH_DATA[c_]["cat"],
                                **{re.sub(r"[^A-Za-z0-9]+", "_", k): np.stack(v) for k, v in BPRED[c_].items()})
    BRUNS = save_table(pd.DataFrame(BRUNS), "b05_rerun_log"); display(BRUNS)

    # ---- scores
    ROWS = []
    for c in BENCH_CITIES:
        D = BENCH_DATA[c]; y, cat, K = D["yte"], D["cat"], D["K"]
        for name, plist in BPRED[c].items():
            for s, p in enumerate(plist):
                p = np.asarray(p, float)
                ROWS.append(dict(city=c, model=name, seed=s, **{f"P1_{k}": v for k, v in p_metrics(p, y, cat, K, y > 0).items()},
                                 **{f"P2_{k}": v for k, v in p_metrics(p, y, cat, K, D["keep2"]).items()}, **{f"P3_{k}": v for k, v in full_metrics(p, y, cat, K).items()}))
    BSC = save_table(pd.DataFrame(ROWS), "b06_benchmark_scores_per_seed")
    num = [c_ for c_ in BSC.columns if c_[:3] in ("P1_", "P2_", "P3_")]
    BMEAN = BSC.groupby(["city", "model"])[num].mean(); BSD = BSC.groupby(["city", "model"])[num].std(); BN = BSC.groupby(["city", "model"]).size().rename("seeds")
    BAGG = BMEAN.join(BSD.add_suffix("_sd")).join(BN).reset_index(); save_table(BAGG, "b07_benchmark_scores_mean_sd")

    # official-protocol table with the published numbers
    OFF = []
    for c in BENCH_CITIES:
        for name, v in PUBLISHED[c].items():
            OFF.append(dict(city=c, model=name, source="published (ST-HSL paper, Table III)", seeds=np.nan, **{f"MAE_{k}": v[2 * k] for k in range(4)}, **{f"MAPE_{k}": v[2 * k + 1] for k in range(4)}))
        for _, r in BAGG[BAGG["city"] == c].iterrows():
            OFF.append(dict(city=c, model=r["model"], source="this notebook (mean over seeds)", seeds=r["seeds"],
                            **{f"MAE_{k}": r[f"P1_MAE_{k}"] for k in range(4)}, **{f"MAPE_{k}": r[f"P1_MAPE_{k}"] for k in range(4)},
                            **{f"MAE_{k}_sd": r[f"P1_MAE_{k}_sd"] for k in range(4)}))
    OFF = pd.DataFrame(OFF); OFF["mean_MAE"] = OFF[[f"MAE_{k}" for k in range(4)]].mean(1); save_table(OFF, "b08_official_protocol_with_published")
    for c in BENCH_CITIES:
        ren = {f"MAE_{k}": f"{n} MAE" for k, n in enumerate(BENCH_CATS[c])} | {f"MAPE_{k}": f"{n} MAPE" for k, n in enumerate(BENCH_CATS[c])}
        print(f"\n{c} – official protocol P1 (crime days only)"); display(OFF[OFF["city"] == c].rename(columns=ren).drop(columns="city").sort_values("mean_MAE").round(4))
    FULLT = BAGG[["city", "model", "seeds", "P3_MAE_all", "P3_RMSE_all", "P3_deviance_all", "P3_deviance_all_sd", "P3_MAE_zero_days", "P3_MAE_crime_days"]]
    print("\nFull protocol P3 (all cell-days)"); display(FULLT.sort_values(["city", "P3_deviance_all"]).round(4))

### I.3 Reproduction gap, metric exploit and ranking stability
- **Reproduction gap**: re-run (mean ± sd over seeds) minus published, per city and category, under the official protocol P1. ST-SHN is shown both ways: as its code reports (best *test* epoch) and with the epoch chosen on validation; the difference is the optimism from selecting on the test set.
- **Metric exploit**: how many published models / categories a constant 1 or the protocol-aware forecast "beats".
- **Rankings**: rank of every model with predictions under P1 (mean MAE over the 4 categories), P2 and P3 (Poisson deviance and MAE on all cell-days), with Kendall's τ between protocols.

In [ ]:
with part("I.3 benchmark: reproduction gap, exploit, rankings, intervals"):
    REPRO = []
    MAP_PUB = {"ST-HSL (re-run)": "ST-HSL", "ST-SHN (re-run, epoch chosen on validation)": "ST-SHN", "ST-SHN (re-run, as its code reports: best test epoch)": "ST-SHN",
               "Graph WaveNet (official model code, re-trained)": "GWN", "MTGNN (official model code, re-trained)": "MTGNN"}
    for c in BENCH_CITIES:
        for rerun, pubname in MAP_PUB.items():
            g = BSC[(BSC["city"] == c) & (BSC["model"] == rerun)]
            if g.empty: continue
            for k, n in enumerate(BENCH_CATS[c]):
                pub_mae, pub_mape = PUBLISHED[c][pubname][2 * k], PUBLISHED[c][pubname][2 * k + 1]
                m, sd = g[f"P1_MAE_{k}"].mean(), g[f"P1_MAE_{k}"].std()
                REPRO.append(dict(city=c, rerun=rerun, published_as=pubname, category=n, seeds=len(g), published_MAE=pub_mae, rerun_MAE=m, rerun_MAE_sd=sd,
                                  gap_MAE=m - pub_mae, gap_pct=100 * (m - pub_mae) / pub_mae, rerun_worse=m > pub_mae,
                                  published_MAPE=pub_mape, rerun_MAPE=g[f"P1_MAPE_{k}"].mean(), gap_MAPE=g[f"P1_MAPE_{k}"].mean() - pub_mape))
    REPRO = save_table(pd.DataFrame(REPRO), "b09_reproduction_gap"); display(REPRO.round(4))
    if not REPRO.empty:
        display(REPRO.groupby(["rerun"]).agg(cells=("gap_MAE", "size"), rerun_worse=("rerun_worse", "sum"), mean_gap_pct=("gap_pct", "mean")).round(2))
    # test-set selection optimism of ST-SHN
    SEL = BSC[BSC["model"].str.startswith("ST-SHN")].groupby(["city", "model"])[["P1_MAE", "P1_MAPE", "P2_MAE", "P2_MAPE", "P3_deviance_all"]].mean().reset_index()
    save_table(SEL, "b10_stshn_test_selection_optimism"); display(SEL.round(4))

    # metric exploit
    EXPL = []
    for c in BENCH_CITIES:
        pub = PUBLISHED[c]
        for name in ("Trivial: constant 1", "Ours – protocol-aware (zero-truncated median)", BENCH_DATA[c]["ours"]):
            g = BSC[(BSC["city"] == c) & (BSC["model"] == name)]
            for k, n in enumerate(BENCH_CATS[c]):
                mae, mape = g[f"P1_MAE_{k}"].mean(), g[f"P1_MAPE_{k}"].mean()
                EXPL.append(dict(city=c, model=name, category=n, MAE=mae, MAPE=mape, published_STHSL_MAE=pub["ST-HSL"][2 * k], published_STHSL_MAPE=pub["ST-HSL"][2 * k + 1],
                                 beats_STHSL_MAE=mae < pub["ST-HSL"][2 * k], beats_STHSL_MAPE=mape < pub["ST-HSL"][2 * k + 1],
                                 published_models_beaten_MAE=sum(v[2 * k] > mae for v in pub.values()), published_models_beaten_MAPE=sum(v[2 * k + 1] > mape for v in pub.values()),
                                 published_models=len(pub)))
    EXPL = save_table(pd.DataFrame(EXPL), "b11_metric_exploit"); display(EXPL.round(4))

    # rankings under the three protocols
    RANK, TAU = [], []
    for c in BENCH_CITIES:
        g = BAGG[BAGG["city"] == c].copy()
        g["P1_mean_MAE"] = g[[f"P1_MAE_{k}" for k in range(4)]].mean(1); g["P2_mean_MAE"] = g[[f"P2_MAE_{k}" for k in range(4)]].mean(1)
        cols = {"P1 official ST-HSL (mean MAE, crime days)": "P1_mean_MAE", "P2 official ST-SHN (mean MAE, subset)": "P2_mean_MAE",
                "P3 full (Poisson deviance, all cell-days)": "P3_deviance_all", "P3 full (MAE, all cell-days)": "P3_MAE_all"}
        for lab, col in cols.items(): g[f"rank: {lab}"] = g[col].rank(method="min")
        RANK.append(g[["city", "model", "seeds"] + list(cols.values()) + [f"rank: {l}" for l in cols]])
        for a_, b_ in (("P1_mean_MAE", "P3_deviance_all"), ("P1_mean_MAE", "P3_MAE_all"), ("P2_mean_MAE", "P3_deviance_all"), ("P1_mean_MAE", "P2_mean_MAE")):
            tau, pv = kendalltau(g[a_], g[b_])
            TAU.append(dict(city=c, protocol_a=a_, protocol_b=b_, kendall_tau=tau, p_value=pv, best_a=g.loc[g[a_].idxmin(), "model"], best_b=g.loc[g[b_].idxmin(), "model"],
                            models=len(g)))
    RANK = save_table(pd.concat(RANK), "b12_rankings_by_protocol"); TAU = save_table(pd.DataFrame(TAU), "b13_ranking_agreement")
    display(RANK.sort_values(["city", "P3_deviance_all"]).round(4)); display(TAU.round(3))

    # simple vs deep on the full protocol, seed by seed (A4)
    A4 = []
    for c in BENCH_CITIES:
        ours = BSC[(BSC["city"] == c) & (BSC["model"] == BENCH_DATA[c]["ours"])]["P3_deviance_all"].values
        for m in [x for x in BSC["model"].unique() if x.startswith(("ST-HSL", "ST-SHN (re-run, epoch chosen", "Graph WaveNet", "MTGNN"))]:
            dv = BSC[(BSC["city"] == c) & (BSC["model"] == m)]["P3_deviance_all"].values
            if len(dv): A4.append(dict(city=c, deep_model=m, seeds=len(dv), deep_deviance_min=dv.min(), deep_deviance_mean=dv.mean(),
                                       ours_deviance_max=ours.max(), ours_deviance_mean=ours.mean(), ours_better_every_seed=bool(ours.max() < dv.min())))
    A4 = save_table(pd.DataFrame(A4), "b14_simple_vs_deep_full_protocol"); display(A4.round(4))

    # 90 % intervals on the benchmark (calibrated on the 30 validation days, our forecaster, seed 0)
    BCOV = []
    for c in BENCH_CITIES:
        D = BENCH_DATA[c]; mu = np.asarray(BPRED[c][D["ours"]][0]); mv = np.asarray(D["val_preds"][D["ours"]][0]); y, yv = D["yte"], D["yva"]
        vol = pd.Series(D["Xall"][:, :, :D["ntr"]].mean(2).reshape(-1)); vq = pd.qcut(vol.rank(method="first"), 5, labels=False).values + 1
        vq_rows = np.tile(vq, D["nte"]); sc = norm_score(yv, mv); edges = mondrian_edges(mv); bv = np.digitize(mv, edges)
        qb = np.array([conformal_q(sc[bv == i]) for i in range(len(edges) + 1)]); r_nb = nb_size(mv, yv)
        meths = {"Poisson": parametric_interval(mu, "Poisson"), "Neg. binomial": parametric_interval(mu, "NB", r_nb),
                 "Split conformal": int_from_q(mu, conformal_q(sc)), "Mondrian conformal": int_from_q(mu, qb[np.digitize(mu, edges)])}
        for meth, (lo, hi) in meths.items(): BCOV.append(dict(city=c, method=meth, **coverage_report(y, lo, hi, vq_rows)))
    BCOV = save_table(pd.DataFrame(BCOV), "b15_benchmark_interval_coverage"); display(BCOV.round(3))

    # figures
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    for i, c in enumerate(BENCH_CITIES):
        d = OFF[OFF["city"] == c].sort_values("mean_MAE")
        col = ["#c0392b" if m.startswith("Ours:") else "#e67e22" if ("constant" in m or "protocol-aware" in m) else "#27ae60" if ("re-run" in m or "re-trained" in m)
               else "#7f8c8d" if s_.startswith("published") else "#2980b9" for m, s_ in zip(d["model"], d["source"])]
        ax[i].barh(d["model"] + np.where(d["source"].str.startswith("published"), " [pub.]", ""), d["mean_MAE"], color=col); ax[i].invert_yaxis()
        ax[i].set_title(f"{c}-Crimes: official protocol (crime days only), mean MAE over 4 categories\ngrey published · green re-run · red ours · orange metric exploit · blue naive", fontsize=9)
        ax[i].tick_params(axis="y", labelsize=7)
    savefig("figB1_official_protocol_published_vs_rerun")
    if not REPRO.empty:
        fig, ax = plt.subplots(figsize=(11, 4.5)); g = REPRO.copy(); g["label"] = g["city"] + " " + g["category"]
        for j, (m, gg) in enumerate(g.groupby("rerun")): ax.scatter(gg["gap_pct"], gg["label"], label=m, s=40, marker="osD^v"[j % 5])
        ax.axvline(0, color="k", lw=.8); ax.set_xlabel("re-run minus published official MAE (%)"); ax.legend(fontsize=7); ax.set_title("Reproduction gap (positive = re-run worse than published)")
        savefig("figB2_reproduction_gap")
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    for i, c in enumerate(BENCH_CITIES):
        g = RANK[RANK["city"] == c]
        for _, r in g.iterrows():
            ax[i].plot([0, 1], [r["rank: P1 official ST-HSL (mean MAE, crime days)"], r["rank: P3 full (Poisson deviance, all cell-days)"]], marker="o")
            ax[i].text(1.03, r["rank: P3 full (Poisson deviance, all cell-days)"], r["model"][:45], fontsize=7, va="center")
        ax[i].set_xticks([0, 1]); ax[i].set_xticklabels(["official (crime days)", "full (deviance)"]); ax[i].invert_yaxis(); ax[i].set_xlim(-.1, 1.9)
        ax[i].set_ylabel("rank (1 = best)"); ax[i].set_title(f"{c}: model ranks change with the protocol")
    savefig("figB3_rank_change_by_protocol")

## Summary: hypothesis verdicts and results file
Verdicts are computed from the saved tables with the thresholds fixed at the top of the notebook. `results_summary.md` collects the verdicts and the key tables; everything is zipped into `/kaggle/working/optionA_outputs.zip`.

In [ ]:
def g_(name):
    v = globals().get(name); return v if isinstance(v, pd.DataFrame) and len(v) else None
def verdict(ok, partly=False): return "supported" if ok else ("partly supported" if partly else "not supported")
HYP = []
with part("VI. hypothesis verdicts and summary"):
    R_ = g_("REPRO")
    if R_ is not None:
        ev = []; oks = []
        for m in ("ST-HSL (re-run)", "ST-SHN (re-run, epoch chosen on validation)"):
            gg = R_[R_["rerun"] == m]
            if len(gg): oks.append(gg["rerun_worse"].sum() >= 6); ev.append(f"{m}: re-run worse in {int(gg['rerun_worse'].sum())}/{len(gg)} cells, mean gap {gg['gap_pct'].mean():+.1f} %")
        HYP.append(dict(hypothesis="A1", claim="re-runs of ST-HSL and ST-SHN are worse than published in ≥ 6/8 cells",
                        verdict=verdict(len(oks) == 2 and all(oks), any(oks)) if oks else "not evaluated", evidence="; ".join(ev) if ev else "no ST-HSL / ST-SHN re-run finished"))
    else: HYP.append(dict(hypothesis="A1", claim="reproduction gap", verdict="not evaluated", evidence="benchmark part did not run"))
    E_ = g_("EXPL")
    if E_ is not None:
        c1 = E_[E_["model"] == "Trivial: constant 1"].groupby("city")["beats_STHSL_MAPE"].sum(); z = E_[E_["model"].str.contains("protocol-aware")].groupby("city")["beats_STHSL_MAE"].sum()
        ok = (c1 > 0).all() and (z > 0).all() and len(c1) == 2
        HYP.append(dict(hypothesis="A2", claim="a constant 1 beats published ST-HSL MAPE, and the protocol-aware forecast beats its MAE, in ≥ 1 category per city", verdict=verdict(ok, (c1 > 0).any()),
                        evidence="; ".join(f"{c}: constant-1 beats MAPE in {int(c1.get(c, 0))}/4, protocol-aware beats MAE in {int(z.get(c, 0))}/4" for c in BENCH_CITIES)))
    T_ = g_("TAU")
    if T_ is not None:
        t1 = T_[(T_["protocol_a"] == "P1_mean_MAE") & (T_["protocol_b"] == "P3_deviance_all")]
        ok = ((t1["kendall_tau"] < 0.8) | (t1["best_a"] != t1["best_b"])).any()
        HYP.append(dict(hypothesis="A3", claim="rankings disagree between the official and the full protocol (τ < 0.8 or a different winner)", verdict=verdict(ok),
                        evidence="; ".join(f"{r.city}: τ = {r.kendall_tau:.2f}, best official = {r.best_a}, best full = {r.best_b}" for r in t1.itertuples())))
    A4_ = g_("A4")
    if A4_ is not None:
        HYP.append(dict(hypothesis="A4", claim="our forecaster has lower full-protocol deviance than every re-run deep model, in every seed", verdict=verdict(A4_["ours_better_every_seed"].all(), A4_["ours_better_every_seed"].any()),
                        evidence="; ".join(f"{r.city} vs {r.deep_model.split(' (')[0]}: {r.ours_deviance_mean:.3f} vs {r.deep_deviance_mean:.3f}" for r in A4_.itertuples())))
    S_ = g_("SKS")
    if S_ is not None:
        nv = S_[S_["baseline_family"] == "naive/statistical"]
        HYP.append(dict(hypothesis="A5", claim="the selected forecaster beats every naive/statistical baseline at H = 1–4 in all cities and seeds", verdict=verdict(nv["significant_all_seeds"].all(), nv["significant_all_seeds"].mean() > 0.5),
                        evidence=f"{int(nv['significant_all_seeds'].sum())}/{len(nv)} horizon × city × baseline cells significant in every seed; selected = {RATE_MODELS[O3_SEL]}"))
    C_ = g_("COV")
    if C_ is not None:
        c0 = C_[C_["seed"] == C_.groupby(["H", "city"])["seed"].transform("min")]
        mon = c0[c0["method"] == "Mondrian CP"]; others = {m: c0[c0["method"] == m] for m in ("Split conformal (CP)", "Poisson", "Neg. binomial")}
        ok_m = (mon["max_abs_dev_from_90"] <= 0.03).all(); ok_o = all((g["max_abs_dev_from_90"] > 0.03).any() for g in others.values())
        HYP.append(dict(hypothesis="A6", claim="Mondrian keeps every volume quintile within 3 points of 90 % (all cities, horizons); CP, Poisson, NB do not", verdict=verdict(ok_m and ok_o, ok_m),
                        evidence=f"Mondrian worst deviation {100 * mon['max_abs_dev_from_90'].max():.1f} pts; " + "; ".join(f"{m} worst {100 * g['max_abs_dev_from_90'].max():.1f} pts" for m, g in others.items())))
    RC_ = g_("ROLL_COV")
    if RC_ is not None:
        mon = RC_[RC_["method"] == "Mondrian CP"]; ok1 = (mon["max_abs_dev_from_90"] <= 0.03).all()
        a20 = RC_[(RC_["test_year"] == 2020) & RC_["method"].isin(["ACI", "Mondrian-ACI"])]
        ok2 = len(a20) > 0 and (a20.groupby("city")["coverage"].apply(lambda s: (s - 0.9).abs().min()) <= 0.02).all()
        HYP.append(dict(hypothesis="A7", claim="A6 holds in every rolling test year; in 2020 ACI brings coverage within 2 points of 90 %", verdict=verdict(ok1 and ok2, ok1 or ok2),
                        evidence=f"Mondrian worst deviation over folds {100 * mon['max_abs_dev_from_90'].max():.1f} pts (years {[int(v) for v in sorted(mon['test_year'].unique())]}); "
                                 + "2020 ACI coverage: " + ", ".join(f"{r.city} {r.method} {100 * r.coverage:.1f} %" for r in a20.itertuples())))
    GP_ = g_("GAP")
    if GP_ is not None:
        gp = GP_[GP_["grouping"] == "poverty_group"].pivot_table(index="city", columns="method", values="gap")
        if {"Mondrian CP", "Split conformal (CP)"} <= set(gp.columns):
            better = gp["Mondrian CP"].abs() < gp["Split conformal (CP)"].abs()
            HYP.append(dict(hypothesis="A8", claim="Mondrian has a smaller high- vs low-poverty coverage gap than CP in every city", verdict=verdict(better.all(), better.any()),
                            evidence="; ".join(f"{c}: Mondrian {100 * gp.loc[c, 'Mondrian CP']:+.1f} vs CP {100 * gp.loc[c, 'Split conformal (CP)']:+.1f} pts" for c in gp.index)))
    if not any(h["hypothesis"] == "A8" for h in HYP): HYP.append(dict(hypothesis="A8", claim="equity of coverage", verdict="not evaluated", evidence="census data or intervals unavailable"))
    HYP = save_table(pd.DataFrame(HYP), "z01_hypothesis_verdicts"); display(HYP)

    def md(df, n=60):
        try: return df.head(n).to_markdown(index=False, floatfmt=".4f")
        except Exception: return df.head(n).to_string(index=False)
    STATUS["VI. hypothesis verdicts and summary"] = "ok"
    lines = [f"# Option A experiments – results summary", f"Generated {pd.Timestamp.now():%Y-%m-%d %H:%M}, run time {hours_used():.2f} h, FAST = {FAST}.", "",
             "## Part status", md(pd.DataFrame([dict(part=k, status=v) for k, v in STATUS.items()])), "", "## Hypotheses", md(HYP), ""]
    for title, name, cols in [("Benchmark: official protocol vs published (mean over seeds)", "OFF", None), ("Benchmark: full protocol", "FULLT", None),
                              ("Benchmark: reproduction gap", "REPRO", None), ("Benchmark: ranking agreement", "TAU", None), ("Benchmark: protocol audit", "AUDIT", None),
                              ("Benchmark: re-run log", "BRUNS", None), ("Weekly panels", "DS", None), ("Model selection (validation)", None, None),
                              ("Forecast skill summary", "SKS", None), ("Interval coverage (main split)", "COV", None), ("Rolling-origin coverage", "ROLL_COV", None),
                              ("Rolling-origin skill", "ROLL_SKILL", None), ("Coverage gap between groups", "GAP", None), ("Forecast skill by group", "EQ_SKILL", None)]:
        d = g_(name) if name else None
        if title.startswith("Model selection") and "vsel" in globals(): d = vsel.rename("validation_deviance").reset_index()
        if d is not None: lines += [f"## {title}", md(d, 120), ""]
    (OUT / "results_summary.md").write_text("\n".join(lines))
    json.dump(dict(status=STATUS, hours=hours_used(), versions=VERSIONS, hypotheses=HYP.to_dict("records")), open(OUT / "metrics_summary.json", "w"), indent=1, default=str)
    log("results_summary.md written")
print(f"Finished in {hours_used():.2f} h. Download /kaggle/working/optionA_outputs.zip from the Output tab.")